# Monitoreo de fauna silvestre con cámaras trampa (YOLO) — Guía paso a paso en Google Colab

**Dataset:** Caltech Camera Traps (CCT-20, LILA BC) · **Modelo:** YOLOv8 (Ultralytics) · **Entorno:** Google Colab con GPU

Este cuaderno tiene dos partes:

* **PARTE A — Plan general y metodología** (texto): qué se hace, en qué orden, con qué decisiones y por qué.
* **PARTE B — Código explicado** (celdas): cada bloque tiene primero una explicación de *qué hace y por qué*, y luego el código con comentarios. Se ejecuta de arriba hacia abajo.

> **Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware: **T4 GPU*** (o mejor si tienes Colab Pro). Luego ejecuta las celdas en orden. Las celdas marcadas ⏱ son largas (descarga, entrenamiento, evaluación).

### Ruta de lectura rápida
| Entregable del documento | Dónde se resuelve en este cuaderno |
|---|---|
| 1. Modelo YOLO + decisiones documentadas (resolución, aumento, imágenes vacías) | Secciones 4 y 5 |
| 2. mAP@0.5 y mAP@0.5:0.95 por especie + efecto del desbalance | Sección 6 |
| 3. Generalización entre ubicaciones (cis vs trans) + evidencia de memorización del fondo (Grad-CAM) | Sección 7 |
| 4. Robustez ante oclusión y tamaño del objeto | Sección 8 |
| 5. Módulo de monitoreo ecológico + demostración | Sección 9 |
| 6. Discusión de uso responsable de IA en conservación | Sección 10 |

---
# PARTE A — PLAN GENERAL Y METODOLOGÍA DE TRABAJO

## A.1 Objetivo
Entrenar y evaluar un detector YOLO que **localice y clasifique la especie** de los animales en fotos de cámaras trampa, medir **cuánto se degrada** al pasar a cámaras nuevas, y construir un **módulo de reglas** que convierta las detecciones en un reporte de actividad por cámara, especie y periodo.

## A.2 Aclaraciones importantes sobre los datos (léelas antes de empezar)
El enunciado describe el dataset **completo** (243 100 imágenes, 140 ubicaciones, 21 categorías, ~70 % vacías). Pero el experimento del punto 3 y el entrenamiento de un detector (que necesita *bounding boxes*) se hacen con el **subconjunto de referencia CCT-20** que LILA publica para ese fin. Estas son las diferencias que debes declarar en tu informe:

| Aspecto | Enunciado (dataset completo) | CCT-20 (el que usa este cuaderno) |
|---|---|---|
| Imágenes | 243 100 | ≈ 57 900 (reducidas a máx. 1024 px por lado, ~6 GB) |
| Ubicaciones de cámara | 140 | 20 (10 de entrenamiento —reutilizadas por `cis_val`/`cis_test`—, 1 en `trans_val` y 9 en `trans_test`) |
| Categorías | 21 + vacío | ≈ 15 especies/objetos (+ vacío); el cuaderno las lee del JSON |
| Fracción de vacías | ≈ 70 % | **0 % en `train`** (el `train` oficial no trae vacías) y ≈ 8–9 % de las imágenes utilizables en test. Además hay ≈ 1 450 imágenes de `train` con animal pero **sin caja** (se excluyen). Los fondos se obtienen de las vacías de `cis_val` (ver 5.2) |
| Partición oficial | “eccv_train / eccv_val” | `train`, `cis_val`, `cis_test`, `trans_val`, `trans_test` (benchmark ECCV 2018 de Beery et al.) |

* **cis** = *mismas ubicaciones* que en entrenamiento (pero otros días) → mide generalización “fácil”.
* **trans** = *ubicaciones nunca vistas* → mide generalización geográfica/fondo nuevo (el problema central del enunciado).
* Lo que el enunciado llama “eccv_train / eccv_val” corresponde a esta partición ECCV. El cuaderno **no asume nombres de archivo**: busca los JSON, los imprime y comprueba que no haya fuga entre particiones (sección 3).
* Fuentes de las cifras: página del dataset en LILA BC (https://lila.science/datasets/caltech-camera-traps) y el artículo del benchmark (Beery et al., *Recognition in Terra Incognita*, ECCV 2018). **Verifica los conteos con la salida de la sección 3** y reporta los *tuyos*.
## A.3 Metodología por fases
| Fase | Qué se hace | Salida | Esfuerzo orientativo* |
|---|---|---|---|
| F0 | Entorno Colab (GPU, Drive, semillas, versiones fijas) | Reproducibilidad | 0.5 h |
| F1 | Descarga CCT-20, lectura de anotaciones, **chequeos anti-fuga**, EDA (desbalance, tamaños, vacías, ubicaciones) | Tablas/figuras de EDA | 1–2 h |
| F2 | Conversión COCO→YOLO, **imágenes vacías como fondo**, verificación visual de etiquetas | Dataset YOLO + `data.yaml` | 1 h |
| F3 | Entrenamiento del modelo base (E0) con decisiones documentadas | `best.pt`, curvas | 1.5–4 h de GPU |
| F4 | Evaluación: mAP50 y mAP50-95 global y **por especie**, análisis de desbalance | Tabla por especie, gráficos | 0.5 h |
| F5 | **Generalización cis vs trans**: brecha, AP por ubicación, FP en vacías, **Grad-CAM** y prueba de “animal enmascarado” | Brecha + evidencia | 1–2 h |
| F6 | **Robustez**: oclusión (proxy bajo/medio/alto) y tamaño relativo | Tabla de caídas | 1–2 h |
| F7 | **Módulo de monitoreo ecológico** + demo en una cámara no vista | Reporte tabular (CSV/Excel) | 1–2 h |
| F8 | Experimentos opcionales (E1–E3), discusión ética, informe final | Informe + paquete de entregables | 3–6 h |

\* Los tiempos de GPU dependen de la GPU que Colab te asigne; **mide la primera época y extrapola**.

## A.4 Decisiones de diseño y su justificación
| Decisión | Elección por defecto | Por qué |
|---|---|---|
| Modelo | **YOLOv8s** preentrenado en COCO (alternativa: `yolov9s.pt`) | Buen equilibrio precisión/velocidad en una T4; el enunciado recomienda v8/v9. El preentrenamiento ayuda mucho con ~13 k imágenes. |
| Resolución de entrada | **640** (E0). Ablación opcional a **960** (E3) | 640 es el estándar y entrena rápido; los animales pequeños/lejanos pierden píxeles al reducir, por eso se prueba 960. |
| Imágenes vacías | El `train` oficial **no tiene** vacías → se añaden como **fondos** (etiqueta vacía) las ≈ 1 000 vacías de `cis_val`. Ablación opcional **sin fondos** (E1) | Un detector que nunca ve fondos puros dispara falsos positivos con el viento y la vegetación. Se mide la tasa de FP en vacías del test. |
| Aumento de datos | Mosaic + MixUp suave + HSV fuerte (luz/IR) + volteo horizontal + escala/traslación moderadas; sin rotación ni volteo vertical | Mosaic y MixUp **mezclan fondos de distintas cámaras** (frena la memorización de un fondo); HSV cubre día/noche; las cámaras son fijas y los animales no están boca abajo. |
| Desbalance entre especies | E0 sin re-muestreo (para *medir* el efecto). E2 opcional con **repeat-factor sampling** (estilo LVIS) | El enunciado pide señalar el efecto del desbalance: primero se mide, luego se mitiga. |
| Selección del checkpoint | `val` = `trans_val` (`CFG.val_splits`) | `cis_val` comparte ráfagas con `train` (por diseño del benchmark), así que premiaría memorizar. **Las métricas finales se reportan en `cis_test` y `trans_test`, que nunca se usan para decidir nada.** |
| Evaluación | Ultralytics (mAP50, mAP50-95) + pycocotools para subgrupos | Ultralytics para cifras oficiales; pycocotools permite filtrar por oclusión/tamaño/ubicación. |
| Oclusión | **Proxy** (borde + solape + desorden de textura) validado con una muestra etiquetada a mano | CCT no trae etiquetas de oclusión; hay que **declarar** que es una estimación. |

## A.5 Diseño experimental
| ID | Descripción | Obligatorio | Responde a |
|---|---|---|---|
| **E0** | YOLOv8s, 640 px, con vacías, sin re-muestreo | **Sí** | Entregables 1–5 |
| E1 | E0 sin los fondos añadidos (solo `train` oficial) | Opcional | Efecto del fondo (entregable 1) |
| E2 | E0 + repeat-factor sampling de especies raras | Opcional | Mitigación del desbalance (entregable 2) |
| E3 | E0 a 960 px | Opcional | Animales pequeños (entregable 4) |
| Ext. | Segundo dataset de LILA BC (otra región) | Opcional | Generalización entre datasets |

## A.6 Reglas anti-fuga (rigor experimental)
1. Las ubicaciones de `trans_*` **no** aparecen en `train` (se verifica por código).
2. Las ráfagas (`seq_id`) no se reparten entre particiones (se verifica). Excepción de diseño: `cis_val` comparte ráfagas con `train`, por eso no se usa para elegir el modelo ni como evidencia de generalización.
3. `test` solo se usa **una vez** al final; los umbrales del módulo de monitoreo se fijan con *val*, no con test.
4. Semilla fija (42), versiones de librerías fijas y `decisiones_entrenamiento.json` guardado junto a cada corrida.

## A.7 Riesgos y mitigaciones
| Riesgo | Mitigación |
|---|---|
| Colab desconecta la sesión | Todo se guarda en **Google Drive** (`runs/`); el entrenamiento se reanuda con `resume=True`. |
| Descarga lenta/bloqueada | El cuaderno prueba 2 espejos (GCP y Azure); si fallan, descarga a mano desde LILA y súbelo a Drive. |
| Pocas ejemplos en especies raras → AP ruidoso | Se muestra `n_test` y se marca `n_test < 30` como no concluyente. |
| Falsa sensación de oclusión medida | Validación manual de la muestra y declaración explícita de que es un proxy. |
| Memoria RAM (2 CPU / ~12 GB en Colab gratis) | `workers=2`, `cache=False`, imágenes pre-redimensionadas a 640 px de lado mayor. |

---
# PARTE B — CÓDIGO EXPLICADO

## 1. Preparar el entorno (F0)

**Qué hace:** comprueba que hay GPU, instala las librerías con versión fija (reproducibilidad), monta Google Drive (para no perder resultados si la sesión se cae) y define la **configuración central** `CFG`. Todo lo que quieras cambiar (modelo, resolución, épocas…) está en esa celda.

In [ ]:
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    out = ''
print(out if out else '⚠️ No se detectó GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU')

In [ ]:
# Versión fija de Ultralytics = la que se usó para probar este cuaderno.
!pip -q install "ultralytics==8.4.190" pycocotools scipy openpyxl

In [ ]:
import os, sys, json, math, random, shutil, time, zipfile, tarfile, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict
from types import SimpleNamespace as NS
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2, torch, yaml
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50); pd.set_option('display.width', 200)

# ---------- Reproducibilidad ----------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------- Rutas ----------
# ROOT = disco local rápido de Colab (se borra al cerrar sesión): datos y dataset YOLO.
# OUT  = Google Drive (persistente): corridas, tablas, figuras, predicciones.
ROOT = Path('/content')
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/cct20_yolo')
except ImportError:
    IN_COLAB = False
    OUT = ROOT / 'cct20_yolo_outputs'

RAW    = ROOT / 'cct_raw'            # descarga original
RUNS   = OUT / 'runs'                # entrenamientos y evaluaciones de Ultralytics
TABLES = OUT / 'tablas'              # CSV para el informe
FIGS   = OUT / 'figuras'             # PNG para el informe
PREDS  = OUT / 'preds'               # predicciones guardadas (.npy)
for d in (RAW, RUNS, TABLES, FIGS, PREDS): d.mkdir(parents=True, exist_ok=True)

# ---------- Configuración central (cámbiala aquí) ----------
CFG = NS(
    model      = 'yolov8s.pt',   # alternativa: 'yolov9s.pt'
    imgsz      = 640,            # resolución de entrada (E0). Ablación E3: 960
    max_side   = 640,            # pre-redimensiona imágenes a este lado mayor (acelera la carga). Poner 960 para E3
    epochs     = 50,             # baja a 30 si Colab te limita las horas de GPU
    batch      = 32,             # T4 16 GB: 32 va bien con yolov8s a 640. Si da error CUDA OOM, baja a 16
    workers    = 2,              # Colab tiene 2 CPU
    patience   = 15,             # parada temprana
    conf_pred  = 0.01,           # umbral bajo para guardar predicciones y calcular AP
    iou_nms    = 0.7,
    max_det    = 50,
    conf_op    = 0.25,           # umbral "operativo" (el que usaría un equipo de manejo)
    gradcam_layers = (4, 6, 9),  # capas del BACKBONE para Grad-CAM (YOLOv8: 4/6/9 alimentan las salidas P3/P4/P5)
    bg_from_cis_val = True,      # añade como fondos las imágenes vacías de cis_val (el train oficial no tiene vacías)
    val_splits = ('trans_val',), # validación para elegir best.pt. cis_val comparte secuencias con train (ver 3), por eso no se usa
    non_animal = ('car',),       # clases que NO cuentan como fauna en el monitoreo
)
DEVICE   = 0 if torch.cuda.is_available() else 'cpu'
USE_HALF = bool(torch.cuda.is_available())
HALF_KW  = dict(quantize=16) if USE_HALF else {}      # inferencia en FP16 solo si hay GPU
print('Dispositivo:', DEVICE, '| Torch', torch.__version__, '| Drive/OUT:', OUT)

## 2. Descargar CCT-20 (F1) ⏱

**Qué hace:** descarga (a) las **imágenes del benchmark CCT-20** (~6 GB, ya reducidas a 1024 px) y (b) los **archivos de anotaciones** con la partición oficial (`train`, `cis_val`, `cis_test`, `trans_val`, `trans_test`). Prueba varios espejos. Si ya están descargados, no repite.

> Si ninguna URL funciona (LILA cambia rutas a veces): entra a https://lila.science/datasets/caltech-camera-traps, sección *CCT-20 benchmark*, descarga las imágenes y los metadatos, súbelos a `/content/cct_raw/` con los nombres `images.tar.gz` y `annotations.zip` y vuelve a ejecutar esta celda.

In [ ]:
IMG_URLS = [
    'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_all_images_sm.tar.gz',
    'https://lilablobssc.blob.core.windows.net/caltechcameratraps/eccv_18_all_images_sm.tar.gz',
]
ANN_URLS = [
    'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotation_files.zip',
    'https://lilablobssc.blob.core.windows.net/caltechcameratraps/eccv_18_annotation_files.zip',
]
IMG_TAR, ANN_ZIP = RAW / 'images.tar.gz', RAW / 'annotations.zip'
IMG_DIR, ANN_DIR = RAW / 'images', RAW / 'annotations'

def download(urls, dest, min_bytes=1_000_000):
    """Descarga con reintento/continuación (wget -c) probando cada espejo."""
    if dest.exists() and dest.stat().st_size > min_bytes:
        print(f'✔ ya existe {dest.name} ({dest.stat().st_size/1e9:.2f} GB)'); return
    for u in urls:
        print('Descargando', u)
        r = subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', str(dest), u])
        if r.returncode == 0 and dest.exists() and dest.stat().st_size > min_bytes:
            return
    raise RuntimeError('No se pudo descargar. Descarga manualmente desde LILA y súbelo como ' + str(dest))

if not any(IMG_DIR.rglob('*.jpg')) if IMG_DIR.exists() else True:
    download(IMG_URLS, IMG_TAR)
    IMG_DIR.mkdir(exist_ok=True)
    print('Extrayendo imágenes (puede tardar unos minutos)...')
    subprocess.run(['tar', '-xzf', str(IMG_TAR), '-C', str(IMG_DIR)], check=True)
    IMG_TAR.unlink()          # libera ~6 GB de disco
if not any(ANN_DIR.rglob('*.json')) if ANN_DIR.exists() else True:
    download(ANN_URLS, ANN_ZIP)
    ANN_DIR.mkdir(exist_ok=True)
    zipfile.ZipFile(ANN_ZIP).extractall(ANN_DIR)

print('Imágenes .jpg:', sum(1 for _ in IMG_DIR.rglob('*.jpg')))
print('Archivos JSON:', [p.name for p in ANN_DIR.rglob('*.json')])

## 3. Leer anotaciones, unificar y **verificar que no hay fuga** (F1)

**Qué hace:**
1. Localiza los 5 JSON de la partición (sin asumir su ruta exacta).
2. Construye **una fila por imagen** con: partición, ubicación de cámara, secuencia (ráfaga), fecha y las cajas.
3. Clasifica cada imagen como `animal` (tiene cajas), `empty` (vacía) o `unlabeled` (tiene etiqueta de animal pero sin caja → se excluye para no enseñarle al modelo que ese animal es “fondo”).
4. Lee el **tamaño real** de cada archivo y escala las cajas si el JSON se refería a otra resolución (comprobación de seguridad).
5. **Chequeos anti-fuga:** las ubicaciones `trans_*` no pueden estar en `train`; las secuencias no deben repartirse entre particiones.

> Resultado clave: la tabla de conteos. **Esos** son los números que debes citar en tu informe.

In [ ]:
SPLITS = ['train', 'cis_val', 'cis_test', 'trans_val', 'trans_test']

def find_ann_files(ann_dir):
    """Busca <split>_*.json sin asumir la subcarpeta."""
    found = {}
    for p in sorted(Path(ann_dir).rglob('*.json')):
        n = p.stem.lower()
        for s in SPLITS:
            if n.startswith(s) and len(n) > len(s) and n[len(s)] == '_' and s not in found:
                found[s] = p
    return found

ann_files = find_ann_files(ANN_DIR)
print({k: v.name for k, v in ann_files.items()})
assert set(ann_files) == set(SPLITS), f'Faltan particiones: {set(SPLITS) - set(ann_files)}. Revisa los nombres impresos arriba.'

# Índice nombre_de_archivo -> ruta real (robusto a subcarpetas)
img_index = {p.name: p for p in IMG_DIR.rglob('*.jpg')}

def load_coco_split(split, jpath):
    d = json.load(open(jpath))
    cats = {c['id']: c['name'] for c in d['categories']}
    by_img = defaultdict(list)
    for a in d['annotations']:
        by_img[a['image_id']].append(a)
    rows, missing = [], 0
    for im in d['images']:
        p = img_index.get(Path(im['file_name']).name)
        if p is None:
            missing += 1; continue
        boxes, has_empty, nobox = [], False, 0
        for a in by_img.get(im['id'], []):
            name = cats[a['category_id']]
            if name == 'empty':
                has_empty = True
            elif a.get('bbox'):
                x, y, w, h = a['bbox']; boxes.append((name, x, y, w, h))
            else:
                nobox += 1
        n_ann = len(by_img.get(im['id'], []))
        # animal = tiene caja. empty = etiquetada 'empty'. unlabeled = etiqueta de ANIMAL pero SIN caja:
        # en CCT-20 hay ~1.4 k así en train; NO son vacías (contienen animales). Se excluyen de entrenamiento y evaluación.
        status = 'animal' if boxes else ('unlabeled' if nobox > 0 else 'empty')
        rows.append(dict(split=split, image_id=im['id'], fname=p.name, raw_path=str(p),
                         Wj=im.get('width'), Hj=im.get('height'),
                         location=str(im.get('location')), seq_id=str(im.get('seq_id')),
                         frame_num=im.get('frame_num'),
                         date=im.get('date_captured') or im.get('datetime'),
                         boxes_json=boxes, status=status, n_ann=n_ann))
    if missing: print(f'⚠️ {split}: {missing} imágenes del JSON no están en disco')
    return pd.DataFrame(rows)

df_all = pd.concat([load_coco_split(s, ann_files[s]) for s in SPLITS], ignore_index=True)
df_all['uid'] = np.arange(1, len(df_all) + 1)          # id único global (empieza en 1: pycocotools lo requiere)

# Tamaño real de cada archivo (por si el JSON hablaba de otra resolución)
sizes = [Image.open(p).size for p in tqdm(df_all.raw_path, desc='leyendo tamaños')]
df_all['W0'] = [s[0] for s in sizes]; df_all['H0'] = [s[1] for s in sizes]
df_all['Wj'] = df_all.Wj.fillna(df_all.W0); df_all['Hj'] = df_all.Hj.fillna(df_all.H0)
print('Escala JSON→archivo (debería ser 1.0):', (df_all.W0 / df_all.Wj).round(3).value_counts().head(3).to_dict())

# Nombres de clase (sin "empty"), índice estable alfabético
class_names = sorted({b[0] for bs in df_all.boxes_json for b in bs})
NC = len(class_names)
cls_id = {n: i for i, n in enumerate(class_names)}
print(f'{NC} clases:', class_names)

# ---------- Chequeos anti-fuga ----------
locs = {s: set(df_all[df_all.split == s].location) for s in SPLITS}
seqs = {s: set(df_all[df_all.split == s].seq_id) for s in SPLITS}
print('\nUbicaciones por partición:', {s: len(v) for s, v in locs.items()})
for s in ['trans_val', 'trans_test']:
    ov = locs[s] & locs['train']
    print(f'  {"✅" if not ov else "⚠️"} {s} ∩ train (ubicaciones) = {len(ov)}')
for s in ['cis_val', 'cis_test']:
    print(f'  ℹ️ {s}: {len(locs[s] & locs["train"])}/{len(locs[s])} ubicaciones también están en train (esperado: todas)')
print(f'  {"✅" if not (locs["trans_val"] & locs["trans_test"]) else "ℹ️"} trans_val ∩ trans_test (ubicaciones) = {len(locs["trans_val"] & locs["trans_test"])}')
for a in SPLITS:
    for b in SPLITS:
        if a < b and (seqs[a] & seqs[b]):
            if {a, b} == {'train', 'cis_val'}:
                print(f'  ℹ️ secuencias compartidas train/cis_val: {len(seqs[a] & seqs[b])} → ESPERADO: en el benchmark oficial cis_val se saca de los mismos días de train. '
                      'Por eso cis_val NO es independiente: no lo uses para elegir el modelo ni como prueba de generalización.')
            else:
                print(f'  ⚠️ secuencias compartidas entre {a} y {b}: {len(seqs[a] & seqs[b])}')
print('  (un ⚠️ de secuencias indicaría fuga real; train/cis_val es la excepción de diseño)')

resumen = (df_all.groupby('split')
           .agg(imagenes=('uid', 'size'), ubicaciones=('location', 'nunique'),
                con_fauna=('status', lambda s: (s == 'animal').sum()),
                vacias=('status', lambda s: (s == 'empty').sum()),
                animal_sin_caja=('status', lambda s: (s == 'unlabeled').sum()))
           .reindex(SPLITS))
resumen['pct_vacias'] = (100 * resumen.vacias / (resumen.con_fauna + resumen.vacias)).round(1)      # % de vacías entre las imágenes UTILIZABLES
resumen.to_csv(TABLES / '01_resumen_particiones.csv')
resumen

### 3.0 Verificar las imágenes "animal sin caja" (¡importante!)
Hay imágenes que en el JSON tienen etiqueta de **animal** (opossum, mapache…) pero **ninguna caja**. En `train` son ~1 450. **No son vacías**: tienen animal sin localizar. Por eso el notebook las **excluye** (si las usáramos como fondo, le enseñaríamos al modelo que ese animal es “nada”; si las usáramos en test, sus detecciones correctas contarían como falsos positivos).
Consecuencia: **el `train` oficial no trae ninguna imagen vacía.** Para poder entrenar con fondos explícitos, la sección 4 añade como fondos las vacías de `cis_val` (mismas cámaras que `train`, y `cis_val` ya no se usa para elegir el modelo).
La celda siguiente muestra 12 imágenes “animal sin caja”: deben contener un animal (a veces pequeño o parcial).

In [ ]:
cand = df_all[df_all.status == 'unlabeled']
print(f'{len(cand)} imágenes con etiqueta de animal pero sin caja (excluidas)')
if len(cand):
    sm = cand.sample(min(12, len(cand)), random_state=SEED)
    fig, axes = plt.subplots(3, 4, figsize=(18, 10))
    for ax, r in zip(axes.ravel(), sm.itertuples()):
        ax.imshow(cv2.cvtColor(cv2.imread(r.raw_path), cv2.COLOR_BGR2RGB)); ax.axis('off'); ax.set_title(f'{r.split} · loc {r.location}', fontsize=9)
    plt.tight_layout(); plt.show()

### 3.1 Análisis exploratorio (EDA)

**Qué hace:** cuantifica el problema que describe el enunciado: desbalance entre especies, fracción de vacías, tamaño de los animales y número de imágenes por ubicación. **Guarda las figuras** para el informe. Mira especialmente (a) cuántos ejemplos tienen las especies más raras en `train` y (b) la distribución del tamaño relativo de las cajas (te ayudará a justificar los grupos de la sección 8).

In [ ]:
# Tabla "una fila por caja"
bx = pd.DataFrame([(r.split, r.uid, r.location, b[0], b[3] * b[4] / (r.Wj * r.Hj))
                   for r in df_all.itertuples() for b in r.boxes_json],
                  columns=['split', 'uid', 'location', 'species', 'rel_area'])

sp_tab = bx.pivot_table(index='species', columns='split', values='uid', aggfunc='size', fill_value=0).reindex(columns=SPLITS)
sp_tab['train_imgs'] = bx[bx.split == 'train'].groupby('species').uid.nunique()
sp_tab = sp_tab.fillna(0).astype(int).sort_values('train', ascending=False)
sp_tab.to_csv(TABLES / '02_cajas_por_especie_y_particion.csv')
display(sp_tab)

tr = sp_tab.train[sp_tab.train > 0]
print(f'Razón de desbalance en train (más común / más rara): {tr.max()} / {tr.min()} = {tr.max()/tr.min():.0f}×')

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
tr.sort_values().plot.barh(ax=ax[0], color='#3b7dd8'); ax[0].set_xscale('log')
ax[0].set_title('Cajas por especie en TRAIN (escala log)'); ax[0].set_xlabel('n.º de cajas')
ax[1].hist(np.log10(bx.rel_area), bins=60, color='#3b7dd8')
ax[1].set_title('Tamaño relativo de la caja (log10 área/área imagen)'); ax[1].set_xlabel('log10(área relativa)')
for q in [0.01, 0.05]:
    ax[1].axvline(np.log10(q), color='r', ls='--'); ax[1].text(np.log10(q), ax[1].get_ylim()[1]*0.9, f'{q:.0%}', color='r')
loc_n = df_all.groupby(['split', 'location']).size().reset_index(name='n')
for s, c in zip(SPLITS, ['#3b7dd8', '#7fb3f5', '#9ecae1', '#e6550d', '#fdae6b']):
    sub = loc_n[loc_n.split == s].sort_values('n', ascending=False)
    ax[2].bar([f'{s}:{l}' for l in sub.location], sub.n, color=c, label=s)
ax[2].set_xticks([]); ax[2].set_title('Imágenes por ubicación de cámara (color = partición)'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / '01_eda.png', dpi=130); plt.show()

print('Cuantiles del área relativa de las cajas:', bx.rel_area.quantile([.1, .25, .5, .75, .9]).round(4).to_dict())

## 4. Convertir a formato YOLO (F2)

**Qué hace:**
* Crea la estructura que Ultralytics espera: `images/<partición>/*.jpg` y `labels/<partición>/*.txt` (una línea por animal: `clase cx cy w h`, normalizada a 0-1).
* **Imágenes vacías = imágenes de fondo:** se copian con un `.txt` **vacío**. Como el `train` oficial no trae vacías, se añaden las de `cis_val` (carpeta `train_bg`). Ultralytics las usa como ejemplos negativos (“aquí no hay nada”). Esta es la forma explícita de manejar el fondo como clase en un detector YOLO (no existe una clase “vacío”; el fondo es la ausencia de cajas).
* Se **excluyen** las imágenes `unlabeled` (animal sin caja, ≈ 1 450 en train: **no son vacías**).
* Si `CFG.max_side` está definido, se **redimensionan** las imágenes a ese lado mayor (acelera mucho el entrenamiento con 2 CPU; las etiquetas normalizadas no cambian).
* Se crean los `data*.yaml`: uno de entrenamiento (`val` = `cis_val` + `trans_val`) y uno por cada partición de evaluación.

La función devuelve un objeto `ds` con el DataFrame final (`ds.df`), donde las cajas ya están en **píxeles de la imagen final** (columna `boxes`, lista de `(clase, x, y, w, h)`).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def _write_image(src, dst, size):
    """Hardlink si no hay que redimensionar (instantáneo, no duplica disco); si no, redimensiona."""
    if dst.exists(): return
    if size is None:
        try: os.link(src, dst)
        except OSError: shutil.copy2(src, dst)
    else:
        im = cv2.imread(str(src)); im = cv2.resize(im, size, interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst), im, [cv2.IMWRITE_JPEG_QUALITY, 95])

def build_dataset(name, max_side):
    root = ROOT / 'cct_yolo' / name
    d = df_all[df_all.status != 'unlabeled'].copy()
    print('Imágenes usadas:', len(d), '| excluidas (sin caja):', (df_all.status == 'unlabeled').sum())

    # tamaño final por imagen
    scale = np.ones(len(d)) if not max_side else np.minimum(1.0, max_side / np.maximum(d.W0, d.H0))
    d['W'] = np.round(d.W0 * scale).astype(int); d['H'] = np.round(d.H0 * scale).astype(int)
    d['resize'] = scale < 1.0

    for s in SPLITS:
        (root / 'images' / s).mkdir(parents=True, exist_ok=True); (root / 'labels' / s).mkdir(parents=True, exist_ok=True)
    d['ypath'] = [str(root / 'images' / s / f) for s, f in zip(d.split, d.fname)]

    # 1) imágenes
    jobs = [(r.raw_path, Path(r.ypath), (r.W, r.H) if r.resize else None) for r in d.itertuples()]
    with ThreadPoolExecutor(4) as ex:
        list(tqdm(ex.map(lambda a: _write_image(*a), jobs), total=len(jobs), desc='imágenes'))

    # 2) etiquetas YOLO + cajas en píxeles finales
    boxes_px, n_drop = [], 0
    for r in tqdm(d.itertuples(), total=len(d), desc='etiquetas'):
        lines, bl = [], []
        for (name_, x, y, w, h) in r.boxes_json:
            x, y, w, h = x * r.W / r.Wj, y * r.H / r.Hj, w * r.W / r.Wj, h * r.H / r.Hj   # a píxeles de la imagen final
            x1, y1, x2, y2 = max(0, x), max(0, y), min(r.W, x + w), min(r.H, y + h)     # recorta a la imagen
            if (x2 - x1) < 2 or (y2 - y1) < 2: n_drop += 1; continue                     # caja degenerada
            bl.append((cls_id[name_], x1, y1, x2 - x1, y2 - y1))
            lines.append(f'{cls_id[name_]} {(x1+x2)/2/r.W:.6f} {(y1+y2)/2/r.H:.6f} {(x2-x1)/r.W:.6f} {(y2-y1)/r.H:.6f}')
        boxes_px.append(bl)
        # imagen vacía => archivo .txt VACÍO (= fondo)
        (root / 'labels' / r.split / (Path(r.fname).stem + '.txt')).write_text('\n'.join(lines))
    d['boxes'] = boxes_px
    # si al recortar se quedó sin cajas, pasa a 'empty'? No: se descarta para no enseñar un animal como fondo
    bad = (d.status == 'animal') & (d.boxes.apply(len) == 0)
    print('Cajas degeneradas descartadas:', n_drop, '| imágenes con animal que quedaron sin cajas (se quitan):', int(bad.sum()))
    d = d[~bad].reset_index(drop=True)

    # 2b) FONDOS: el train oficial no trae imágenes vacías. Se toman las vacías de cis_val (mismas cámaras de train; cis_val no se usa para elegir el modelo ni como test)
    n_bg = 0
    if CFG.bg_from_cis_val:
        for sub_ in ('images', 'labels'): (root / sub_ / 'train_bg').mkdir(parents=True, exist_ok=True)
        for r in d[(d.split == 'cis_val') & (d.status == 'empty')].itertuples():
            di = root / 'images' / 'train_bg' / r.fname
            if not di.exists():
                try: os.link(r.ypath, di)
                except OSError: shutil.copy2(r.ypath, di)
            (root / 'labels' / 'train_bg' / (Path(r.fname).stem + '.txt')).write_text('')      # .txt vacío = fondo
            n_bg += 1
    print(f'Fondos añadidos al entrenamiento (vacías de cis_val): {n_bg}')

    # 3) yaml
    names = {i: n for i, n in enumerate(class_names)}
    def wy(fname, train, val):
        p = root / fname
        yaml.safe_dump(dict(path=str(root), train=train, val=val, names=names), open(p, 'w'), sort_keys=False)
        return p
    vals = [f'images/{s}' for s in CFG.val_splits]
    ymls = dict(train=wy('data.yaml', ['images/train', 'images/train_bg'] if n_bg else 'images/train', vals),
                train_only=wy('data_train_only.yaml', 'images/train', vals))
    for s in ['cis_val', 'trans_val', 'cis_test', 'trans_test']:
        ymls[s] = wy(f'data_{s}.yaml', 'images/train', f'images/{s}')
    ymls['test_all'] = wy('data_test_all.yaml', 'images/train', ['images/cis_test', 'images/trans_test'])
    ds = NS(name=name, root=root, df=d, yaml=ymls, n_bg=n_bg)
    d.drop(columns=['boxes_json']).to_pickle(OUT / f'dataset_{name}.pkl')
    return ds

ds = build_dataset('s%d' % CFG.max_side if CFG.max_side else 'orig', CFG.max_side)
print(ds.df.groupby(['split', 'status']).size().unstack(fill_value=0).reindex(SPLITS))

### 4.1 Verificación visual de las etiquetas (¡no te la saltes!)

**Qué hace:** dibuja las cajas leyendo **los archivos `.txt` YOLO** (no el DataFrame) sobre 8 imágenes aleatorias de entrenamiento. Si las cajas no encajan con los animales, hay un error de formato/escala y todo lo posterior sería inválido.

In [ ]:
def draw_yolo(img_path, ax, title=''):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    lab = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
    for ln in lab.read_text().splitlines():
        c, cx, cy, w, h = ln.split(); c = int(c); cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
        cv2.rectangle(img, (int(cx-w/2), int(cy-h/2)), (int(cx+w/2), int(cy+h/2)), (255, 40, 40), 3)
        cv2.putText(img, class_names[c], (int(cx-w/2), max(15, int(cy-h/2)-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)
    ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis('off')

samp = ds.df[ds.df.split == 'train'].sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, r in zip(axes.ravel(), samp.itertuples()):
    draw_yolo(r.ypath, ax, f'{r.status} · loc {r.location}')
plt.tight_layout(); plt.savefig(FIGS / '02_verificacion_etiquetas.png', dpi=110); plt.show()

## 5. Entrenar el modelo base E0 (F3) ⏱

### 5.1 Decisiones de entrenamiento (Entregable 1)
Todas quedan escritas en `decisiones_entrenamiento.json` junto a los pesos. Resumen y justificación:

| Parámetro | Valor | Justificación |
|---|---|---|
| Modelo | `yolov8s.pt` (COCO) | Transferencia de aprendizaje: con ~13 k imágenes entrenar desde cero rinde peor. |
| `imgsz` | 640 | Estándar; `max_side=640` evita redimensionar “al vuelo”. Ablación a 960 en E3. |
| Imágenes vacías | ≈ 1 000 fondos (vacías de `cis_val`) con etiqueta vacía | El `train` oficial no trae vacías; sin fondos el modelo dispara falsas alarmas. Ablación E1. |
| `mosaic=1.0`, `close_mosaic=10` | activo, apagado en las 10 últimas épocas | Mezcla 4 imágenes → mezcla **fondos de cámaras distintas** (frena la memorización del fondo) y simula objetos parciales/pequeños. Las últimas épocas sin mosaico afinan con imágenes “reales”. |
| `mixup=0.1` | suave | Superpone dos escenas; otra forma de romper la asociación animal↔fondo. |
| `hsv_h/s/v = 0.015/0.7/0.4` | fuertes en S y V | Hay fotos diurnas en color y nocturnas en infrarrojo (casi grises); variaciones de luz extremas. |
| `fliplr=0.5`, `flipud=0` | volteo horizontal sí, vertical no | Un animal no aparece boca abajo. |
| `degrees=0`, `scale=0.5`, `translate=0.1` | sin rotación; escala/traslación moderadas | Cámaras fijas; la escala simula animales más lejanos/cercanos. |
| Optimizador | SGD, `lr0=0.01`, cosine LR, `weight_decay=5e-4`, 3 épocas de warm-up | Configuración estándar y estable de YOLOv8 para fine-tuning. |
| `epochs=50`, `patience=15` | con parada temprana | Presupuesto realista en Colab. |
| `val` | `trans_val` | `cis_val` comparte ráfagas con `train`, así que no sirve para elegir el checkpoint; `trans_val` es una cámara nueva. (Limitación: es una sola cámara.) |

### 5.2 Imágenes vacías: cómo se manejan y cómo se mide
* **Hallazgo sobre los datos:** el `train` oficial de CCT-20 **no contiene imágenes vacías** (las ≈ 1 450 sin caja son animales sin localizar, no fondos). Si no hacemos nada, el modelo nunca vería un fondo puro y daría falsas alarmas con viento o vegetación.
* **Decisión:** añadir como **imágenes de fondo** (etiqueta vacía) las ≈ 1 000 imágenes vacías de `cis_val`. Son de las mismas cámaras de `train`, y `cis_val` no se usa para elegir el modelo ni como test. Representan ≈ 8 % del entrenamiento (rango habitual recomendado por Ultralytics: 0–10 %). Es una **desviación de la partición oficial que debes declarar** en el informe.
* **Cómo se mide:** tasa de falsos positivos en las **vacías de test** (`cis_test` ≈ 1 180, `trans_test` ≈ 1 780) con el umbral operativo (sección 7).
* **Ablación E1 (opcional):** entrenar solo con el `train` oficial (sin fondos) y comparar esa tasa de FP.

### 5.3 Código

**Qué hace esta celda:** define `train_experiment`, que (1) guarda las decisiones, (2) **reanuda** automáticamente si la sesión de Colab se cayó (busca `last.pt`) y (3) marca la corrida como terminada. Ultralytics guarda en Drive, así que no pierdes el avance.

In [ ]:
from ultralytics import YOLO

def train_experiment(name, data_yaml, imgsz=None, model_name=None, **over):
    run_dir = RUNS / name; run_dir.mkdir(parents=True, exist_ok=True)
    last, best, done = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt', run_dir / 'FIN.txt'
    if done.exists():
        print(f'✔ {name} ya entrenado → {best}'); return best

    args = dict(data=str(data_yaml), imgsz=imgsz or CFG.imgsz, epochs=CFG.epochs, batch=CFG.batch,
                workers=CFG.workers, patience=CFG.patience, device=DEVICE, project=str(RUNS), name=name, exist_ok=True,
                seed=SEED, pretrained=True, amp=True, cache=False, plots=True,
                # optimización
                optimizer='SGD', lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=5e-4, warmup_epochs=3, cos_lr=True,
                # aumento de datos
                mosaic=1.0, close_mosaic=10, mixup=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5)
    args.update(over)
    json.dump({'model': model_name or CFG.model, **args}, open(run_dir / 'decisiones_entrenamiento.json', 'w'), indent=2)

    if last.exists():                       # ¿se cayó la sesión? → reanudar
        print('↻ Reanudando desde', last)
        try:
            YOLO(str(last)).train(resume=True)
        except Exception as e:
            print('No se pudo reanudar (¿ya estaba completo?):', repr(e)[:200])
    else:
        YOLO(model_name or CFG.model).train(**args)
    done.write_text(time.strftime('%Y-%m-%d %H:%M:%S'))
    return best

In [ ]:
# ⏱ Mide la duración de la 1.ª época en el log y extrapola al total.
BEST = train_experiment('E0_base', ds.yaml['train'])
print('Pesos:', BEST)

### 5.4 Curvas de entrenamiento

**Qué mirar:** (a) si la pérdida de validación deja de bajar mientras la de entrenamiento sigue (sobreajuste), (b) la diferencia entre mAP de validación y lo que verás luego en `trans_test`.

In [ ]:
res = pd.read_csv(RUNS / 'E0_base' / 'results.csv'); res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(res.epoch, res['train/box_loss'], label='train box'); ax[0].plot(res.epoch, res['val/box_loss'], label='val box')
ax[0].plot(res.epoch, res['train/cls_loss'], '--', label='train cls'); ax[0].plot(res.epoch, res['val/cls_loss'], '--', label='val cls')
ax[0].set_title('Pérdidas'); ax[0].legend()
ax[1].plot(res.epoch, res['metrics/mAP50(B)'], label='mAP50'); ax[1].plot(res.epoch, res['metrics/mAP50-95(B)'], label='mAP50-95')
ax[1].set_title('mAP en validación (' + ' + '.join(CFG.val_splits) + ')'); ax[1].legend()
ax[2].plot(res.epoch, res['metrics/precision(B)'], label='precisión'); ax[2].plot(res.epoch, res['metrics/recall(B)'], label='recall')
ax[2].set_title('Precisión / recall (val)'); ax[2].legend()
plt.tight_layout(); plt.savefig(FIGS / '03_curvas_entrenamiento.png', dpi=130); plt.show()

## 6. Evaluación cuantitativa y efecto del desbalance (Entregable 2) ⏱

**Qué hace:**
* Evalúa `best.pt` con el evaluador de Ultralytics en las 4 particiones y en el **test combinado** (`cis_test` + `trans_test`), con umbral de confianza 0.001 (el estándar para calcular AP).
* Devuelve **mAP@0.5** y **mAP@0.5:0.95** globales y **por especie**.
* Cruza el AP de cada especie con su **número de ejemplos de entrenamiento** para mostrar el efecto del desbalance: correlación de Spearman y comparación entre especies *raras*, *intermedias* y *comunes* (terciles de `n_train`).

> **Cómo leerlo:** el mAP global promedia las especies por igual (*macro*), así que unas pocas especies raras con AP bajo lo arrastran. Fíjate en `n_test`: con pocas cajas de prueba el AP es muy ruidoso y no es concluyente (se marca con ⚠️).

In [ ]:
def ultra_val(weights, yaml_path, tag, imgsz=None, plots=False):
    """Evalúa con Ultralytics. Devuelve métricas globales y DataFrame por especie."""
    model = YOLO(str(weights))
    m = model.val(data=str(yaml_path), imgsz=imgsz or CFG.imgsz, batch=CFG.batch, conf=0.001, iou=CFG.iou_nms,
                  device=DEVICE, **HALF_KW, plots=plots, project=str(RUNS / 'eval'), name=tag, exist_ok=True,
                  verbose=False, workers=CFG.workers)
    b = m.box
    glob = dict(P=float(b.mp), R=float(b.mr), mAP50=float(b.map50), mAP50_95=float(b.map))
    per = pd.DataFrame({'species': [class_names[int(i)] for i in b.ap_class_index],
                        'AP50': b.all_ap[:, 0], 'AP50_95': b.all_ap.mean(1), 'P': b.p, 'R': b.r})
    return glob, per, Path(m.save_dir)

def evaluate_all(weights, ds_, exp, imgsz=None):
    G, PER = {}, {}
    for s in ['cis_val', 'trans_val', 'cis_test', 'trans_test', 'test_all']:
        G[s], PER[s], sd = ultra_val(weights, ds_.yaml[s], f'{exp}_{s}', imgsz, plots=(s == 'test_all'))
        if s == 'test_all': SAVE_DIR = sd
    return G, PER, SAVE_DIR

In [ ]:
G, PER, SAVE_DIR = evaluate_all(BEST, ds, 'E0')
glob_tab = pd.DataFrame(G).T[['mAP50', 'mAP50_95', 'P', 'R']].round(4)
glob_tab.to_csv(TABLES / '03_metricas_globales_E0.csv'); glob_tab

In [ ]:
# Conteos de ejemplos por especie
def count_boxes(split_list):
    d = ds.df[ds.df.split.isin(split_list)]
    c = Counter(b[0] for bl in d.boxes for b in bl)
    return pd.Series({n: c.get(i, 0) for i, n in enumerate(class_names)})
def count_imgs(split_list):
    d = ds.df[ds.df.split.isin(split_list)]
    c = Counter(i for bl in d.boxes for i in {b[0] for b in bl})
    return pd.Series({n: c.get(i, 0) for i, n in enumerate(class_names)})

sp = pd.DataFrame({'n_train_cajas': count_boxes(['train']), 'n_train_imgs': count_imgs(['train']),
                   'n_test_cis': count_boxes(['cis_test']), 'n_test_trans': count_boxes(['trans_test'])})
for s in ['cis_test', 'trans_test', 'test_all']:
    t = PER[s].set_index('species')[['AP50', 'AP50_95']].add_suffix('_' + s.replace('_test', '').replace('test_all', 'todo'))
    sp = sp.join(t)
sp['n_test_todo'] = sp.n_test_cis + sp.n_test_trans
sp['aviso'] = np.where(sp.n_test_todo < 30, '⚠️ n_test<30 (ruidoso)', '')
sp = sp.sort_values('n_train_cajas', ascending=False)
sp.round(3).to_csv(TABLES / '04_ap_por_especie_E0.csv')
display(sp.round(3))

# ---- Efecto del desbalance ----
from scipy.stats import spearmanr
v = sp.dropna(subset=['AP50_95_todo']).copy()
rho, pval = spearmanr(np.log10(v.n_train_cajas.clip(lower=1)), v.AP50_95_todo)
v['grupo'] = pd.qcut(v.n_train_cajas.rank(method='first'), 3, labels=['rara', 'intermedia', 'común'])
grp = v.groupby('grupo').agg(especies=('AP50_95_todo', 'size'), n_train_med=('n_train_cajas', 'median'),
                             AP50=('AP50_todo', 'mean'), AP50_95=('AP50_95_todo', 'mean')).round(3)
grp.to_csv(TABLES / '05_ap_por_grupo_de_frecuencia.csv')
print(f'Spearman(log n_train, AP50-95) = {rho:.2f}  (p = {pval:.3f}, n = {len(v)} especies)')
display(grp)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].scatter(v.n_train_cajas, v.AP50_95_todo, s=60, c='#3b7dd8')
for n, r in v.iterrows(): ax[0].annotate(n, (r.n_train_cajas, r.AP50_95_todo), fontsize=8, xytext=(4, 3), textcoords='offset points')
ax[0].set_xscale('log'); ax[0].set_xlabel('n.º de cajas de entrenamiento (log)'); ax[0].set_ylabel('AP@0.5:0.95 (test)')
ax[0].set_title(f'Desbalance vs desempeño (Spearman ρ={rho:.2f})')
o = v.sort_values('n_train_cajas'); ax[1].barh(o.index, o.AP50_95_todo, color='#3b7dd8')
ax[1].set_xlabel('AP@0.5:0.95 (test) — ordenado de rara (arriba) a común (abajo)'); ax[1].set_title('AP por especie')
plt.tight_layout(); plt.savefig(FIGS / '04_desbalance_vs_ap.png', dpi=130); plt.show()

**Matriz de confusión** (generada por Ultralytics sobre el test combinado): muestra *con qué especie* se confunden las especies raras y cuánto se pierde hacia/desde el fondo (“background”): la última fila/columna son los **falsos negativos** (animal no detectado) y **falsos positivos** (detección sobre el fondo).

In [ ]:
from IPython.display import Image as IPImage, display
for f in ['confusion_matrix_normalized.png', 'confusion_matrix.png']:
    p = SAVE_DIR / f
    if p.exists(): display(IPImage(str(p), width=700)); break

### 6.1 Experimentos opcionales E1, E2, E3 (ablaciones del Entregable 1/2) ⏱

Cámbialos a `True` solo si tienes tiempo de GPU. Cada uno reutiliza `train_experiment`.
* **E1 – sin fondos:** entrena solo con el `train` oficial, sin las vacías de `cis_val`. ¿Sube la tasa de falsos positivos sobre las vacías de test? (se mide en la sección 7).
* **E2 – repeat-factor sampling:** las imágenes con especies raras se repiten más veces: `r = máx(1, √(t / f_c))` con `f_c` = fracción de imágenes de train que contienen la especie `c` (como en LVIS), tope 4.
* **E3 – 960 px:** reconstruye el dataset a 960 px y entrena a 960 (mejor para animales pequeños, más lento). Requiere `batch` más pequeño (p. ej. 16).

In [ ]:
RUN_E1, RUN_E2, RUN_E3 = False, False, False      # ← pon True lo que quieras ejecutar

def make_variant(ds_, kind='rfs', t=0.05, cap=4):
    """Crea train_rfs/ con hardlinks (no duplica disco): repite más las imágenes con especies raras."""
    src = ds_.df[ds_.df.split == 'train']
    vname = f'train_{kind}'
    for sub_ in ('images', 'labels'): (ds_.root / sub_ / vname).mkdir(parents=True, exist_ok=True)
    f = Counter(c for bl in src.boxes for c in {b[0] for b in bl})
    f = {c: n / len(src) for c, n in f.items()}
    rc = {c: max(1.0, math.sqrt(t / fc)) for c, fc in f.items()}
    print('repeat factor por especie:', {class_names[c]: round(v, 2) for c, v in sorted(rc.items())})
    n = 0
    for r in src.itertuples():
        cs = {b[0] for b in r.boxes}
        reps = 1 if not cs else min(cap, math.ceil(max(rc[c] for c in cs)))
        for k in range(reps):
            stem = Path(r.fname).stem + (f'__r{k}' if k else '')
            di, dl = ds_.root / 'images' / vname / (stem + '.jpg'), ds_.root / 'labels' / vname / (stem + '.txt')
            if not di.exists():
                try: os.link(r.ypath, di)
                except OSError: shutil.copy2(r.ypath, di)
            shutil.copy2(ds_.root / 'labels' / 'train' / (Path(r.fname).stem + '.txt'), dl); n += 1
    y = ds_.root / f'data_{vname}.yaml'
    tr = [f'images/{vname}'] + (['images/train_bg'] if ds_.n_bg else [])          # los fondos se mantienen
    yaml.safe_dump(dict(path=str(ds_.root), train=tr, val=[f'images/{s}' for s in CFG.val_splits],
                        names={i: nm for i, nm in enumerate(class_names)}), open(y, 'w'), sort_keys=False)
    print(f'{vname}: {n} imágenes de entrenamiento con animal (original: {len(src)}) + {ds_.n_bg} fondos'); return y

EXPS = {'E0_base': (BEST, ds, CFG.imgsz)}
if RUN_E1:
    EXPS['E1_sin_fondos'] = (train_experiment('E1_sin_fondos', ds.yaml['train_only']), ds, CFG.imgsz)
if RUN_E2:
    EXPS['E2_rfs'] = (train_experiment('E2_rfs', make_variant(ds)), ds, CFG.imgsz)
if RUN_E3:
    ds960 = build_dataset('s960', 960)
    EXPS['E3_960'] = (train_experiment('E3_960', ds960.yaml['train'], imgsz=960, batch=16), ds960, 960)

if len(EXPS) > 1:
    rows = []
    for k, (w, d_, sz) in EXPS.items():
        for s in ['cis_test', 'trans_test', 'test_all']:
            g, _, _ = ultra_val(w, d_.yaml[s], f'{k}_{s}', sz)
            rows.append(dict(exp=k, split=s, **{m: round(v, 4) for m, v in g.items()}))
    cmp_tab = pd.DataFrame(rows).pivot(index='exp', columns='split', values=['mAP50', 'mAP50_95'])
    cmp_tab.to_csv(TABLES / '06_comparacion_experimentos.csv'); display(cmp_tab)
else:
    print('Solo E0 activo. Activa RUN_E1/E2/E3 para comparar.')

## 7. Generalización entre ubicaciones de cámara (Entregable 3) ⏱

**Idea del experimento.** El modelo se entrenó con 10 ubicaciones (`train`).
* `cis_test`: *las mismas ubicaciones* de entrenamiento, otros días → el fondo ya lo “conoce”.
* `trans_test`: ubicaciones **nunca vistas** → el fondo es nuevo.

(Ojo: `cis_val` **no** es una buena vara de medir porque comparte ráfagas con `train`; para “vistas” usa `cis_test`.) Si el rendimiento cae mucho de cis a trans, el modelo se apoyaba en el fondo. Lo cuantificamos y buscamos **evidencia** con cuatro herramientas:

1. **Brecha global** cis→trans en mAP (absoluta y relativa).
2. **AP por ubicación** (¿la brecha es uniforme o depende de unas pocas cámaras difíciles?) + prueba de Mann-Whitney.
3. **Falsos positivos en imágenes vacías** y detección “hay fauna / no hay fauna” a nivel de imagen.
4. **Evidencia directa de reliance en el fondo:** *Grad-CAM* sobre el backbone y la prueba del **“animal enmascarado”**.

### 7.1 Predicciones guardadas y utilidades de evaluación (pycocotools)

**Qué hace:** corre el modelo una sola vez sobre cada partición de prueba y guarda las predicciones (`.npy`, en Drive). Con ellas y **pycocotools** podemos calcular AP para *cualquier subconjunto* (por ubicación, por tamaño, por oclusión), cosa que el evaluador de Ultralytics no permite. Detalle técnico: para evaluar un subgrupo de cajas, las cajas de otros grupos se marcan como *crowd* (se ignoran: una detección sobre ellas no cuenta como falso positivo) y solo se evalúan las imágenes que contienen alguna caja del grupo.

In [ ]:
import contextlib, io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
TDEV = torch.device('cuda:0' if DEVICE != 'cpu' else 'cpu')

def predict_split(weights, d, tag, imgsz=None):
    """Predice en todas las imágenes de d. Devuelve array Nx7: [uid, x, y, w, h, score, clase+1] (px)."""
    f = PREDS / f'{tag}.npy'
    if f.exists(): return np.load(f)
    model = YOLO(str(weights)); uid_by = dict(zip(d.fname, d.uid)); rows = []
    gen = model.predict(source=list(d.ypath), stream=True, conf=CFG.conf_pred, iou=CFG.iou_nms, imgsz=imgsz or CFG.imgsz,
                        batch=CFG.batch, **HALF_KW, max_det=CFG.max_det, device=DEVICE, verbose=False)
    for r in tqdm(gen, total=len(d), desc=f'predict {tag}'):
        b = r.boxes
        if b is None or len(b) == 0: continue
        uid = uid_by[Path(r.path).name]
        for (x1, y1, x2, y2), c, k in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
            rows.append((uid, x1, y1, x2 - x1, y2 - y1, c, k + 1))
    arr = np.array(rows, dtype=np.float64).reshape(-1, 7); np.save(f, arr); return arr

def gt_table(d):
    """Una fila por caja GT (píxeles de la imagen final) con id de anotación 'aid' (empieza en 1)."""
    rows = []
    for r in d.itertuples():
        for (c, x, y, w, h) in r.boxes:
            rows.append(dict(uid=r.uid, split=r.split, location=r.location, cls=c, x=x, y=y, w=w, h=h,
                             W=r.W, H=r.H, rel_area=w * h / (r.W * r.H)))
    g = pd.DataFrame(rows); g.insert(0, 'aid', np.arange(1, len(g) + 1)); return g

def build_coco_gt(d, g):
    ims = [dict(id=int(r.uid), width=int(r.W), height=int(r.H)) for r in d.itertuples()]
    ans = [dict(id=int(r.aid), image_id=int(r.uid), category_id=int(r.cls) + 1, bbox=[r.x, r.y, r.w, r.h],
                area=float(r.w * r.h), iscrowd=0) for r in g.itertuples()]
    gt = COCO(); gt.dataset = dict(images=ims, annotations=ans, categories=[dict(id=i + 1, name=n) for i, n in enumerate(class_names)])
    with contextlib.redirect_stdout(io.StringIO()): gt.createIndex()
    return gt

def _mean_valid(x):
    x = x[x > -1]; return float(x.mean()) if x.size else float('nan')

def coco_eval(gt, dets, img_ids, ignore_aids=()):
    """AP@[.5:.95] y AP@.5 (macro sobre clases con GT) restringido a img_ids. ignore_aids = cajas a ignorar."""
    img_ids = sorted(set(int(i) for i in img_ids))
    sel = dets[np.isin(dets[:, 0], img_ids)]
    if len(sel) == 0: return dict(AP=0.0, AP50=0.0, ap50_c=np.full(NC, np.nan), ap_c=np.full(NC, np.nan))
    ign = [int(a) for a in ignore_aids]
    for a in ign: gt.anns[a]['iscrowd'] = 1
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            dt = gt.loadRes(sel)
            E = COCOeval(gt, dt, 'bbox'); E.params.imgIds = img_ids
            E.params.areaRng = [[0, 1e10]]; E.params.areaRngLbl = ['all']
            E.evaluate(); E.accumulate()
    finally:
        for a in ign: gt.anns[a]['iscrowd'] = 0
    P = E.eval['precision'][:, :, :, 0, 2]              # [IoU, recall, clase] con maxDets=100
    return dict(AP=_mean_valid(P), AP50=_mean_valid(P[0]),
                ap_c=np.array([_mean_valid(P[:, :, k]) for k in range(P.shape[2])]),
                ap50_c=np.array([_mean_valid(P[0, :, k]) for k in range(P.shape[2])]))

def presence_metrics(d, dets, conf=None):
    """Nivel imagen: ¿hay fauna? Devuelve tasa de FP en vacías, recall y precisión de 'imagen con fauna'."""
    conf = conf or CFG.conf_op
    na = [cls_id[n] + 1 for n in CFG.non_animal if n in cls_id]
    k = dets[(dets[:, 5] >= conf) & ~np.isin(dets[:, 6], na)]
    pred = set(k[:, 0].astype(int))
    is_fauna = d.boxes.apply(lambda bl: any((b[0] + 1) not in na for b in bl))
    is_empty = d.status == 'empty'
    uids = d.uid.astype(int)
    p = uids.isin(pred)
    return dict(n_vacias=int(is_empty.sum()), FP_en_vacias=float(p[is_empty].mean()) if is_empty.any() else np.nan,
                recall_fauna=float(p[is_fauna].mean()) if is_fauna.any() else np.nan,
                precision_fauna=float(is_fauna[p].mean()) if p.any() else np.nan)

In [ ]:
TEST = ds.df[ds.df.split.isin(['cis_test', 'trans_test'])].reset_index(drop=True)
DETS = {s: predict_split(BEST, ds.df[ds.df.split == s], f'E0_{s}') for s in ['cis_test', 'trans_test']}
DETS_ALL = np.vstack(list(DETS.values()))
GT_TAB = gt_table(TEST)                         # todas las cajas de test (para secciones 7 y 8)
GT = build_coco_gt(TEST, GT_TAB)
print({k: len(v) for k, v in DETS.items()}, 'detecciones guardadas | cajas GT de test:', len(GT_TAB))

### 7.2 La brecha cis → trans

**Qué hace:** compara mAP en ubicaciones vistas (`cis`) y nuevas (`trans`), para *val* y *test*, con las cifras de Ultralytics (oficiales) y verifica que pycocotools da casi lo mismo (control de calidad de nuestra herramienta de subgrupos; pueden diferir unas milésimas por el umbral 0.01 y el redondeo de la interpolación).

In [ ]:
rows = []
for kind in ['val', 'test']:
    c, t = G[f'cis_{kind}'], G[f'trans_{kind}']
    for m in ['mAP50', 'mAP50_95']:
        rows.append(dict(partición=kind, métrica=m, cis_ubic_vistas=c[m], trans_ubic_nuevas=t[m],
                         brecha_abs=c[m] - t[m], brecha_rel_pct=100 * (c[m] - t[m]) / c[m] if c[m] else np.nan))
gap = pd.DataFrame(rows).round(4); gap.to_csv(TABLES / '07_brecha_cis_trans.csv'); display(gap)

# control cruzado con pycocotools
chk = {}
for s in ['cis_test', 'trans_test']:
    e = coco_eval(GT, DETS[s], ds.df[ds.df.split == s].uid)
    chk[s] = dict(pycoco_AP50=round(e['AP50'], 4), ultra_AP50=round(G[s]['mAP50'], 4),
                  pycoco_AP50_95=round(e['AP'], 4), ultra_AP50_95=round(G[s]['mAP50_95'], 4))
display(pd.DataFrame(chk).T)

# Por especie: ¿qué especies sufren más al cambiar de ubicación?
sp_gap = sp[['n_train_cajas', 'n_test_cis', 'n_test_trans', 'AP50_95_cis', 'AP50_95_trans']].copy()
sp_gap['caida_abs'] = sp_gap.AP50_95_cis - sp_gap.AP50_95_trans
sp_gap.sort_values('caida_abs', ascending=False).round(3).to_csv(TABLES / '08_brecha_por_especie.csv')
display(sp_gap.sort_values('caida_abs', ascending=False).round(3))

### 7.3 AP por ubicación de cámara y falsos positivos en imágenes vacías

**Qué hace:** calcula el AP de **cada cámara** de prueba. Si la brecha viene de pocas cámaras “difíciles” o es general lo dice este gráfico. La prueba de Mann-Whitney compara las dos poblaciones de cámaras (vistas vs nuevas); con ~10 cámaras por grupo úsala como **orientación**, no como prueba definitiva. También calcula, con el umbral operativo (0.25), **qué fracción de imágenes vacías dispara una falsa alarma** y el recall/precisión de “hay fauna” por imagen.

In [ ]:
from scipy.stats import mannwhitneyu
rows = []
for s in ['cis_test', 'trans_test']:
    dd = ds.df[ds.df.split == s]
    for loc, dl in dd.groupby('location'):
        e = coco_eval(GT, DETS[s], dl.uid)
        rows.append(dict(partición=s, ubicación=loc, n_imgs=len(dl), n_cajas=int(sum(len(b) for b in dl.boxes)),
                         AP50=e['AP50'], AP50_95=e['AP'], **presence_metrics(dl, DETS[s])))
loc_tab = pd.DataFrame(rows); loc_tab.round(3).to_csv(TABLES / '09_ap_por_ubicacion.csv'); display(loc_tab.round(3))

a = loc_tab[(loc_tab.partición == 'cis_test') & (loc_tab.n_cajas >= 20)].AP50_95.dropna()
b = loc_tab[(loc_tab.partición == 'trans_test') & (loc_tab.n_cajas >= 20)].AP50_95.dropna()
if len(a) > 2 and len(b) > 2:
    u = mannwhitneyu(a, b, alternative='greater')
    print(f'AP50-95 por cámara — cis: mediana {a.median():.3f} (n={len(a)}) | trans: mediana {b.median():.3f} (n={len(b)}) | Mann-Whitney (cis > trans): p = {u.pvalue:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
o = loc_tab.sort_values(['partición', 'AP50_95'])
ax[0].bar(range(len(o)), o.AP50_95, color=np.where(o.partición == 'cis_test', '#3b7dd8', '#e6550d'))
ax[0].set_xticks(range(len(o))); ax[0].set_xticklabels(o.ubicación, rotation=90, fontsize=7)
ax[0].set_title('AP@0.5:0.95 por cámara (azul = vista en train, naranja = nueva)'); ax[0].set_ylabel('AP')
pm = pd.DataFrame({s: presence_metrics(ds.df[ds.df.split == s], DETS[s]) for s in ['cis_test', 'trans_test']}).T
pm[['FP_en_vacias', 'recall_fauna', 'precision_fauna']].plot.bar(ax=ax[1], rot=0)
ax[1].set_title(f'Nivel imagen @conf≥{CFG.conf_op}: falsas alarmas en vacías, recall y precisión de "hay fauna"')
plt.tight_layout(); plt.savefig(FIGS / '05_generalizacion_por_ubicacion.png', dpi=130); plt.show()
pm.round(3).to_csv(TABLES / '10_presencia_imagen_cis_vs_trans.csv'); display(pm.round(3))

### 7.4 ¿El modelo mira al animal o al fondo? — Grad-CAM sobre el backbone

**Qué es Grad-CAM (explicado simple).** Para una detección concreta (la clase de la especie en la posición donde el modelo está más seguro), calculamos el **gradiente de esa puntuación respecto a los mapas de activación** de una capa del backbone. Promediando el gradiente por canal obtenemos *qué canales importan*, los combinamos con las activaciones y nos queda un **mapa de calor**: zonas de la imagen que más empujan la puntuación de esa especie.

**Cómo lo cuantificamos (no solo “mirar bonitos mapas”):**
* **Energía dentro de la caja** = fracción del mapa de calor que cae dentro de la caja del animal.
* **Enriquecimiento** = energía dentro de la caja ÷ fracción de área que ocupa la caja. *1 = el modelo mira todo por igual (no distingue animal de fondo)*; *>1 = se concentra en el animal*; *<1 = mira más el entorno que al animal*.
* **Pointing game** = ¿el punto más caliente cae dentro de la caja?

Se calcula en `cis_test` y `trans_test`, usando la **clase verdadera** como objetivo. Es una explicación *correlacional*: sugiere en qué se apoya el modelo, no lo prueba causalmente (por eso añadimos la prueba 7.5).

**Elección de la capa (importante):** la cabeza de YOLO predice en 3 escalas (P3, P4, P5: strides 8, 16, 32). Un animal pequeño lo detecta la escala P3, uno grande la P5. Un Grad-CAM tomado en una sola capa fija sería casi nulo para detecciones de otra escala. Por eso, para cada detección **se usa la capa del backbone que alimenta la escala donde ocurrió la detección**: `CFG.gradcam_layers = (4, 6, 9)` en YOLOv8 (salida de los bloques C2f/SPPF del backbone a stride 8, 16 y 32). Si usas YOLOv9 u otra versión, imprime `model.model` y cambia los índices.

In [ ]:
class YoloGradCAM:
    def __init__(self, weights, layers, imgsz):
        self.net = YOLO(str(weights)).model.to(TDEV).float().eval()      # instancia nueva (sin fusionar)
        for p in self.net.parameters(): p.requires_grad_(False)
        self.imgsz, self.acts = imgsz, {}
        for L in layers:                                                  # una capa del backbone por escala P3, P4, P5
            self.net.model[L].register_forward_hook(lambda m, i, o, L=L: self.acts.__setitem__(L, o))
        self.layers = list(layers)

    def __call__(self, img_bgr, target_cls=None):
        h0, w0 = img_bgr.shape[:2]; S = self.imgsz
        r = S / max(h0, w0); nh, nw = round(h0 * r), round(w0 * r)
        top, left = (S - nh) // 2, (S - nw) // 2
        canvas = np.full((S, S, 3), 114, np.uint8); canvas[top:top + nh, left:left + nw] = cv2.resize(img_bgr, (nw, nh))
        x = torch.from_numpy(canvas[..., ::-1].copy()).permute(2, 0, 1).float().div(255)[None].to(TDEV).requires_grad_(True)
        with torch.enable_grad():
            out = self.net(x)
            pred = (out[0] if isinstance(out, (list, tuple)) else out)[0]      # (4+nc, n_anclas)
            sc = pred[4:]                                                      # puntuaciones por clase
            if target_cls is None:
                a = int(sc.max(0).values.argmax()); c = int(sc[:, a].argmax())
            else:
                c = int(target_cls); a = int(sc[c].argmax())
            score = sc[c, a]
            n3, n4 = (S // 8) ** 2, (S // 16) ** 2                             # nº de anclas por escala
            level = 0 if a < n3 else (1 if a < n3 + n4 else 2)                 # ¿en qué escala (P3/P4/P5) está la mejor ancla?
            act = self.acts[self.layers[level]]
            grad = torch.autograd.grad(score, act)[0]                          # d puntuación / d activación
        wts = grad.mean((2, 3), keepdim=True)                                  # peso por canal
        cam = torch.relu((wts * act).sum(1))[0].detach().cpu().numpy()         # mapa de calor
        cam = cv2.resize(cam, (S, S))[top:top + nh, left:left + nw]
        cam = cv2.resize(cam, (w0, h0))
        return cam, dict(cls=c, score=float(score.detach()), escala=f'P{level + 3}')

def cam_metrics(cam, box):
    x, y, w, h = [int(round(v)) for v in box]; H, W = cam.shape
    x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
    tot = cam.sum()
    if tot <= 0: return dict(energia_en_caja=np.nan, enriquecimiento=np.nan, pointing=np.nan, area_frac=np.nan)
    e = cam[y1:y2, x1:x2].sum() / tot; af = (x2 - x1) * (y2 - y1) / (W * H)
    py, px = np.unravel_index(cam.argmax(), cam.shape)
    return dict(energia_en_caja=e, enriquecimiento=e / af, pointing=float(x1 <= px < x2 and y1 <= py < y2), area_frac=af)

def detected(uid, boxrow, dets, conf=None):
    """¿Hay una detección de la clase correcta con IoU>=0.5 y conf>=umbral operativo?"""
    conf = conf or CFG.conf_op
    k = dets[(dets[:, 0] == uid) & (dets[:, 5] >= conf) & (dets[:, 6] == boxrow.cls + 1)]
    for _, x, y, w, h, *_ in k:
        iw = max(0, min(x + w, boxrow.x + boxrow.w) - max(x, boxrow.x)); ih = max(0, min(y + h, boxrow.y + boxrow.h) - max(y, boxrow.y))
        inter = iw * ih
        if inter / (w * h + boxrow.w * boxrow.h - inter + 1e-9) >= 0.5: return True
    return False

# --- Muestra: imágenes con UNA sola caja y animal no gigante (la métrica es limpia) ---
N_CAM = 150
cam_rows, cam_store = [], {}
gcam = YoloGradCAM(BEST, CFG.gradcam_layers, CFG.imgsz)
for s in ['cis_test', 'trans_test']:
    cand = GT_TAB[(GT_TAB.split == s) & (GT_TAB.rel_area < 0.5)]
    cand = cand[cand.uid.map(cand.uid.value_counts()) == 1]                       # una caja por imagen
    cand = cand.sample(min(N_CAM, len(cand)), random_state=SEED)
    for r in tqdm(cand.itertuples(), total=len(cand), desc=f'Grad-CAM {s}'):
        path = ds.df.loc[ds.df.uid == r.uid, 'ypath'].iloc[0]
        img = cv2.imread(path); cam, info = gcam(img, target_cls=r.cls)
        m = cam_metrics(cam, (r.x, r.y, r.w, r.h))
        cam_rows.append(dict(split=s, uid=r.uid, location=r.location, cls=class_names[r.cls], escala=info['escala'], detectado=detected(r.uid, r, DETS[s]), **m))
        if len(cam_store) < 400: cam_store[r.uid] = (path, cam, (r.x, r.y, r.w, r.h))
cam_df = pd.DataFrame(cam_rows); cam_df.round(4).to_csv(TABLES / '11_gradcam_metricas.csv', index=False)

resumen_cam = (cam_df.groupby('split').agg(n=('uid', 'size'), energia_en_caja=('energia_en_caja', 'mean'),
               area_frac_media=('area_frac', 'mean'), enriquecimiento_mediana=('enriquecimiento', 'median'),
               pointing=('pointing', 'mean'), tasa_detectado=('detectado', 'mean')).round(3))
display(resumen_cam)
display(cam_df.groupby(['split', 'detectado']).agg(n=('uid', 'size'), enriquecimiento_mediana=('enriquecimiento', 'median'),
                                                    pointing=('pointing', 'mean')).round(3))
resumen_cam.to_csv(TABLES / '12_gradcam_resumen.csv')

In [ ]:
def overlay(img_bgr, cam, box=None):
    c = cam / (cam.max() + 1e-8)
    heat = cv2.applyColorMap((255 * c).astype(np.uint8), cv2.COLORMAP_JET)
    out = cv2.addWeighted(img_bgr, 0.55, heat, 0.45, 0)
    if box is not None:
        x, y, w, h = [int(v) for v in box]; cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 3)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for row, s in enumerate(['cis_test', 'trans_test']):
    ids = cam_df[cam_df.split == s].uid.tolist()
    ids = [u for u in ids if u in cam_store][:4]
    for ax, u in zip(axes[row], ids):
        path, cam, box = cam_store[u]; img = cv2.imread(path)
        mt = cam_df[cam_df.uid == u].iloc[0]
        ax.imshow(overlay(img, cam, box)); ax.axis('off')
        ax.set_title(f'{s} · {mt.cls}\nenriquec.={mt.enriquecimiento:.2f} · {"detectado" if mt.detectado else "NO detectado"}', fontsize=10)
plt.suptitle('Grad-CAM (backbone). Verde = caja real del animal. Calor = zonas que más apoyan la clase correcta', y=1.0)
plt.tight_layout(); plt.savefig(FIGS / '06_gradcam_galeria.png', dpi=120); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cam_df.boxplot(column='enriquecimiento', by='split', ax=ax[0]); ax[0].axhline(1, color='r', ls='--'); ax[0].set_ylabel('enriquecimiento (1 = sin foco)')
ax[0].set_title('Enriquecimiento del Grad-CAM en la caja'); plt.suptitle('')
cam_df.boxplot(column='energia_en_caja', by='split', ax=ax[1]); ax[1].set_title('Fracción de energía dentro de la caja'); plt.suptitle('')
plt.tight_layout(); plt.savefig(FIGS / '07_gradcam_metricas.png', dpi=120); plt.show()

### 7.5 Prueba causal sencilla: el “animal enmascarado”

**Idea.** Si el modelo reconociera *rasgos del animal*, al **tapar el animal** con un rectángulo gris no debería “ver” esa especie. Si aun así sigue prediciendo la especie original, está usando otras pistas (el fondo/contexto de esa cámara).

* **Condición A (animal enmascarado):** imágenes con un animal, se tapa su caja con gris. Se mide (i) qué fracción sigue produciendo alguna detección (conf ≥ 0.25) y (ii) qué fracción predice **la misma especie** que se tapó.
* **Condición B (control):** imágenes **vacías** de la misma partición con un parche gris del mismo tamaño y posición. Sirve para descontar el efecto del parche en sí.

**Interpretación:** A ≫ B ⇒ hay pistas residuales (fondo/contexto) que bastan para predecir la especie. Compara cis vs trans: en ubicaciones vistas el modelo *podría* “adivinar” la especie por el fondo memorizado; en las nuevas, no.

In [ ]:
def run_masked_test(split, n=200):
    cand = GT_TAB[(GT_TAB.split == split)]
    cand = cand[cand.uid.map(cand.uid.value_counts()) == 1].sample(frac=1, random_state=SEED).head(n)
    empt = ds.df[(ds.df.split == split) & (ds.df.status == 'empty')].sample(frac=1, random_state=SEED)
    model = YOLO(str(BEST)); out = []
    def predict_batch(imgs):
        res = model.predict(source=imgs, conf=CFG.conf_op, iou=CFG.iou_nms, imgsz=CFG.imgsz, **HALF_KW, device=DEVICE, verbose=False)
        return [set(int(c) for c in r.boxes.cls.cpu().numpy()) if r.boxes is not None else set() for r in res]
    imgsA, clsA, imgsB = [], [], []
    for r, e in zip(cand.itertuples(), empt.itertuples()):
        x1, y1, x2, y2 = int(r.x), int(r.y), int(r.x + r.w), int(r.y + r.h)
        a = cv2.imread(ds.df.loc[ds.df.uid == r.uid, 'ypath'].iloc[0]); a[y1:y2, x1:x2] = 114          # tapar animal
        b = cv2.imread(e.ypath); b = cv2.resize(b, (a.shape[1], a.shape[0])); b[y1:y2, x1:x2] = 114   # parche en vacía
        imgsA.append(a); clsA.append(r.cls); imgsB.append(b)
    pa, pb = [], []
    for i in range(0, len(imgsA), 32):
        pa += predict_batch(imgsA[i:i + 32]); pb += predict_batch(imgsB[i:i + 32])
    return dict(partición=split, n=len(imgsA),
                A_cualquier_deteccion=np.mean([len(p) > 0 for p in pa]), A_misma_especie=np.mean([c in p for c, p in zip(clsA, pa)]),
                B_cualquier_deteccion=np.mean([len(p) > 0 for p in pb]))

mk = pd.DataFrame([run_masked_test(s) for s in ['cis_test', 'trans_test']]).round(3)
mk.to_csv(TABLES / '13_prueba_animal_enmascarado.csv', index=False); display(mk)

### 7.6 Cómo redactar la conclusión del Entregable 3
Reúne y contrasta, con **tus** números:
1. Brecha cis→trans en mAP50 y mAP50-95 (tabla 7.2) y si es uniforme entre cámaras (7.3).
2. Aumento de falsas alarmas y caída de recall de “hay fauna” en cámaras nuevas (7.3).
3. Grad-CAM: ¿enriquecimiento >1 y *pointing* alto en ambas particiones (el modelo mira al animal) o cercano a 1/<1 sobre todo en trans (mira el entorno)? Compara además **detectados vs no detectados**.
4. Animal enmascarado: ¿A ≫ B en cis (memorización del fondo) y menor en trans?
5. **Limitaciones**: Grad-CAM es correlacional; la capa elegida es gruesa; hay pocas cámaras por grupo; la brecha mezcla *fondo nuevo* con *otras condiciones* (luz, especies distintas por región, distancias de captura).
**Evita** afirmar “el modelo memoriza el fondo” si las pruebas no lo respaldan: reporta lo que veas, incluso si la evidencia es mixta.

## 8. Robustez ante oclusión y tamaño del objeto (Entregable 4)

**Problema:** CCT **no trae etiquetas de oclusión**. Hay que *estimarla*. Y debemos decirlo claramente en el informe: lo que obtenemos es un **proxy**.

### 8.1 Estimadores por caja
Para cada caja de test calculamos tres señales observables:

| Señal | Qué mide | Cómo |
|---|---|---|
| `trunc` | animal **cortado por el borde** de la foto | nº de lados de la caja pegados al borde (≤2 px), `min(1, lados/2)` |
| `overlap` | animal **tapado por otro animal** | máx. de `área(intersección con otra caja)/área(esta caja)` |
| `clutter` | **vegetación/maleza delante o dentro** de la caja | energía de bordes (Laplaciano) dentro de la caja ÷ la de toda la imagen; luego se convierte a **percentil** |

Se combinan en `occ_score = 0.35·trunc + 0.30·overlap + 0.35·clutter` y se dividen en **terciles** → *bajo / medio / alto* (grupos del mismo tamaño ⇒ comparables). Los pesos son una decisión razonable pero **arbitraria**: por eso en 8.4 se **valida contra etiquetado manual** y se muestran también los grupos por señal individual (`trunc`, `overlap`), que no dependen de pesos.

### 8.2 Tamaño relativo
`rel_area = área de la caja / área de la imagen`, en tres grupos: **pequeño** (< 1.5 %), **mediano** (1.5–5 %) y **grande** (≥ 5 %). Mira el histograma del EDA (sección 3.1) y ajusta `SIZE_BINS` si un grupo queda con menos de ~100 cajas.

### 8.3 Evaluación por grupo
AP50 y AP50-95 por grupo (pycocotools, macro sobre especies presentes), más el **recall de cajas** al umbral operativo (¿qué fracción de animales tiene una detección de su especie con IoU ≥ 0.5?). Se reporta la **caída** entre el grupo más fácil y el más difícil.

In [ ]:
UPATH = dict(zip(ds.df.uid, ds.df.ypath))
rows = []
for uid, g in tqdm(GT_TAB.groupby('uid'), total=GT_TAB.uid.nunique(), desc='características por caja'):
    gray = cv2.imread(UPATH[uid], cv2.IMREAD_GRAYSCALE); H, W = gray.shape
    lap = np.abs(cv2.Laplacian(cv2.GaussianBlur(gray, (3, 3), 0), cv2.CV_32F)); base = lap.mean() + 1e-6
    gl = list(g.itertuples())
    for r in gl:
        x1, y1, x2, y2 = int(r.x), int(r.y), int(r.x + r.w), int(r.y + r.h)
        clutter_raw = float(lap[y1:y2, x1:x2].mean() / base)
        trunc = min(1.0, ((x1 <= 2) + (y1 <= 2) + (x2 >= W - 2) + (y2 >= H - 2)) / 2)
        ov = 0.0
        for o in gl:
            if o.aid == r.aid: continue
            iw = max(0, min(x2, o.x + o.w) - max(x1, o.x)); ih = max(0, min(y2, o.y + o.h) - max(y1, o.y))
            ov = max(ov, iw * ih / (r.w * r.h))
        rows.append((r.aid, clutter_raw, trunc, min(1.0, ov)))
feat = pd.DataFrame(rows, columns=['aid', 'clutter_raw', 'trunc', 'overlap'])
GT_TAB = GT_TAB.drop(columns=[c for c in ['clutter_raw', 'trunc', 'overlap', 'clutter', 'occ_score', 'occ_nivel', 'tam_grupo'] if c in GT_TAB]).merge(feat, on='aid')

GT_TAB['clutter'] = GT_TAB.clutter_raw.rank(pct=True)
GT_TAB['occ_score'] = 0.35 * GT_TAB.trunc + 0.30 * GT_TAB.overlap + 0.35 * GT_TAB.clutter
GT_TAB['occ_nivel'] = pd.qcut(GT_TAB.occ_score.rank(method='first'), 3, labels=['bajo', 'medio', 'alto']).astype(str)

SIZE_BINS = [0, 0.015, 0.05, 1.01]       # según el EDA de CCT-20: ≈25 % / 50 % / 25 % de las cajas
GT_TAB['tam_grupo'] = pd.cut(GT_TAB.rel_area, SIZE_BINS, labels=['pequeño', 'mediano', 'grande'], right=False).astype(str)
GT_TAB['borde'] = np.where(GT_TAB.trunc > 0, 'toca_borde', 'sin_borde')
GT_TAB['solape'] = np.where(GT_TAB.overlap > 0.3, 'solapado>30%', 'sin_solape')

print('Cajas por grupo de oclusión:', GT_TAB.occ_nivel.value_counts().to_dict())
print('Cajas por tamaño:', GT_TAB.tam_grupo.value_counts().to_dict())
print('Tocan el borde:', int((GT_TAB.trunc > 0).sum()), '| solapadas >30 %:', int((GT_TAB.overlap > 0.3).sum()))
GT_TAB.to_csv(TABLES / '14_cajas_test_con_oclusion_y_tamano.csv', index=False)

# Ejemplos visuales de cada nivel (sanidad del proxy)
fig, axes = plt.subplots(3, 6, figsize=(20, 9))
for row, lv in enumerate(['bajo', 'medio', 'alto']):
    sm = GT_TAB[(GT_TAB.occ_nivel == lv) & (GT_TAB.rel_area > 0.01)].sample(6, random_state=SEED)
    for ax, r in zip(axes[row], sm.itertuples()):
        im = cv2.cvtColor(cv2.imread(UPATH[r.uid]), cv2.COLOR_BGR2RGB)
        cv2.rectangle(im, (int(r.x), int(r.y)), (int(r.x + r.w), int(r.y + r.h)), (255, 40, 40), 3)
        ax.imshow(im); ax.axis('off'); ax.set_title(f'oclusión {lv} · {class_names[r.cls]}', fontsize=9)
plt.tight_layout(); plt.savefig(FIGS / '08_ejemplos_oclusion.png', dpi=110); plt.show()

In [ ]:
_o = np.argsort(DETS_ALL[:, 0], kind='stable'); _S = DETS_ALL[_o]
_u, _i = np.unique(_S[:, 0], return_index=True)
DET_BY_UID = {int(u): a for u, a in zip(_u, np.split(_S, _i[1:]))}      # detecciones indexadas por imagen

def box_recall(sub, conf=None):
    """Fracción de cajas GT con ≥1 detección de su clase, conf≥umbral e IoU≥0.5."""
    conf = conf or CFG.conf_op; hit = 0
    for r in sub.itertuples():
        a = DET_BY_UID.get(int(r.uid))
        if a is None: continue
        a = a[(a[:, 5] >= conf) & (a[:, 6] == r.cls + 1)]
        if len(a) == 0: continue
        iw = np.clip(np.minimum(a[:, 1] + a[:, 3], r.x + r.w) - np.maximum(a[:, 1], r.x), 0, None)
        ih = np.clip(np.minimum(a[:, 2] + a[:, 4], r.y + r.h) - np.maximum(a[:, 2], r.y), 0, None)
        inter = iw * ih; iou = inter / (a[:, 3] * a[:, 4] + r.w * r.h - inter + 1e-9)
        hit += bool((iou >= 0.5).any())
    return hit / max(1, len(sub))

def eval_by_group(col, order, min_boxes=30):
    rows = []
    for gname in order:
        for scope in ['test_todo', 'cis_test', 'trans_test']:
            sub = GT_TAB[GT_TAB[col] == gname]
            if scope != 'test_todo': sub = sub[sub.split == scope]
            if len(sub) < min_boxes:
                rows.append(dict(factor=col, grupo=gname, ámbito=scope, n_cajas=len(sub), n_imgs=np.nan, AP50=np.nan, AP50_95=np.nan, recall_cajas=np.nan, nota='n<%d: no evaluado' % min_boxes)); continue
            img_ids = sub.uid.unique()
            ign = GT_TAB.loc[(~GT_TAB.aid.isin(sub.aid)) & GT_TAB.uid.isin(img_ids), 'aid']      # otras cajas de esas imágenes: se ignoran
            e = coco_eval(GT, DETS_ALL, img_ids, ign)
            rows.append(dict(factor=col, grupo=gname, ámbito=scope, n_cajas=len(sub), n_imgs=len(img_ids),
                             AP50=e['AP50'], AP50_95=e['AP'], recall_cajas=box_recall(sub)))
    return pd.DataFrame(rows)

res_occ  = eval_by_group('occ_nivel', ['bajo', 'medio', 'alto'])
res_size = eval_by_group('tam_grupo', ['grande', 'mediano', 'pequeño'])
res_bord = eval_by_group('borde', ['sin_borde', 'toca_borde'])
res_solp = eval_by_group('solape', ['sin_solape', 'solapado>30%'])
ROB = pd.concat([res_occ, res_size, res_bord, res_solp], ignore_index=True)
ROB.round(3).to_csv(TABLES / '15_robustez_por_grupo.csv', index=False); display(ROB.round(3))

def caida(res, mejor, peor, scope='test_todo'):
    a = res[(res.grupo == mejor) & (res.ámbito == scope)].iloc[0]; b = res[(res.grupo == peor) & (res.ámbito == scope)].iloc[0]
    return dict(factor=a.factor, comparación=f'{mejor} → {peor}', ámbito=scope,
                AP50_95_mejor=a.AP50_95, AP50_95_peor=b.AP50_95, caída_abs=a.AP50_95 - b.AP50_95,
                caída_rel_pct=100 * (a.AP50_95 - b.AP50_95) / a.AP50_95, recall_mejor=a.recall_cajas, recall_peor=b.recall_cajas)
cae = []
for scope in ['test_todo', 'cis_test', 'trans_test']:
    for res, m, p in [(res_occ, 'bajo', 'alto'), (res_size, 'grande', 'pequeño'), (res_bord, 'sin_borde', 'toca_borde'), (res_solp, 'sin_solape', 'solapado>30%')]:
        try: cae.append(caida(res, m, p, scope))
        except (IndexError, KeyError, TypeError): pass
cae = pd.DataFrame(cae).round(3); cae.to_csv(TABLES / '16_caidas_de_desempeno.csv', index=False); display(cae)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for a_, (res, order, ttl) in zip(ax, [(res_occ, ['bajo', 'medio', 'alto'], 'Oclusión estimada (proxy)'), (res_size, ['grande', 'mediano', 'pequeño'], 'Tamaño relativo de la caja')]):
    for scope, c in zip(['cis_test', 'trans_test', 'test_todo'], ['#3b7dd8', '#e6550d', '#555555']):
        y = [res[(res.grupo == g) & (res.ámbito == scope)].AP50_95.values for g in order]
        y = [v[0] if len(v) and not pd.isna(v[0]) else np.nan for v in y]
        a_.plot(order, y, marker='o', color=c, label=scope)
    a_.set_title(ttl); a_.set_ylabel('AP@0.5:0.95'); a_.legend()
plt.tight_layout(); plt.savefig(FIGS / '09_robustez.png', dpi=130); plt.show()

### 8.4 Validación manual del proxy de oclusión (recomendado)
**Por qué:** sin esto, “oclusión baja/media/alta” es una etiqueta no verificada. **Qué hacer:** (1) ejecuta la celda: muestra recortes con contexto, **sin** revelar el nivel estimado; (2) mira cada recorte y asigna `0` (visible / casi sin oclusión), `1` (parcialmente tapado) o `2` (muy tapado); (3) escribe tus respuestas en el diccionario `MANUAL` (clave = número `aid` que se ve sobre cada recorte); (4) vuelve a ejecutar la última parte: calcula la correlación de Spearman y la concordancia con el proxy. Con ≥ 60 recortes etiquetados ya es informativo. Si la concordancia es baja, **dilo en el informe** y reemplaza el proxy por tu etiquetado manual (p. ej., 300 cajas).

In [ ]:
_pool = GT_TAB[GT_TAB.rel_area > 0.005]
val_s = pd.concat([g.sample(min(40, len(g)), random_state=SEED) for _, g in _pool.groupby('occ_nivel')]).sample(frac=1, random_state=SEED)
val_s[['aid', 'uid', 'cls', 'x', 'y', 'w', 'h', 'occ_nivel', 'occ_score']].to_csv(TABLES / '17_muestra_validacion_oclusion.csv', index=False)

def show_page(k, per_page=24, cols=6):
    sub = val_s.iloc[k * per_page:(k + 1) * per_page]
    fig, axes = plt.subplots(int(np.ceil(len(sub) / cols)), cols, figsize=(3.6 * cols, 3.0 * int(np.ceil(len(sub) / cols))))
    for ax, r in zip(axes.ravel(), sub.itertuples()):
        im = cv2.cvtColor(cv2.imread(UPATH[r.uid]), cv2.COLOR_BGR2RGB)
        m = int(0.15 * max(r.w, r.h)); y1, y2, x1, x2 = max(0, int(r.y) - m), int(r.y + r.h) + m, max(0, int(r.x) - m), int(r.x + r.w) + m
        cv2.rectangle(im, (int(r.x), int(r.y)), (int(r.x + r.w), int(r.y + r.h)), (255, 40, 40), 2)
        ax.imshow(im[y1:y2, x1:x2]); ax.set_title(f'aid {r.aid}', fontsize=10); ax.axis('off')
    for ax in axes.ravel()[len(sub):]: ax.axis('off')
    plt.tight_layout(); plt.show()

show_page(0)       # cambia a show_page(1), show_page(2)... para ver más recortes
print('Páginas disponibles:', int(np.ceil(len(val_s) / 24)))

In [ ]:
# Rellena con tus etiquetas: {aid: 0/1/2}. Ejemplo:  MANUAL = {1523: 0, 88: 2, 4021: 1}
MANUAL = {}
from scipy.stats import spearmanr
if len(MANUAL) >= 30:
    mv = val_s.set_index('aid').loc[list(MANUAL)].assign(manual=pd.Series(MANUAL))
    rho, p = spearmanr(mv.occ_score, mv.manual)
    print(f'Spearman(occ_score, oclusión manual) = {rho:.2f} (p={p:.3g}, n={len(mv)})')
    display(pd.crosstab(mv.occ_nivel, mv.manual, rownames=['proxy'], colnames=['manual (0/1/2)']))
else:
    print(f'Etiquetas manuales: {len(MANUAL)}/30 mínimas. Completa MANUAL para validar el proxy.')

## 9. Módulo de monitoreo ecológico basado en reglas (Entregable 5)

**Objetivo.** Convertir detecciones por fotograma en lo que necesita un equipo de manejo de fauna: *por cámara*, **cuántas fotos con fauna vs vacías, qué especies, cuántos registros y cuántos eventos independientes**, y *por especie y periodo de tiempo*, la actividad.

### 9.1 Reglas (transparentes y ajustables en `MonitorConfig`)
| Regla | Qué hace | Por qué |
|---|---|---|
| **R1 Confianza por especie** | Una detección cuenta si `conf ≥ umbral` de su especie (por defecto 0.25; se puede **bajar** para especies amenazadas/de interés) | Un FN en una especie amenazada cuesta más que un FP (ver sección 10) |
| **R2 Tamaño mínimo** | Se ignoran cajas con área relativa < `min_rel_area` | Elimina “motitas” (ruido, insectos) |
| **R3 Confirmación por ráfaga** | En una misma ráfaga (`seq_id`; si no existe: fotos separadas ≤10 s) una especie se **confirma** si hay **una detección con `conf ≥ conf_confirm`** *o* **aparece en ≥ `min_frames_confirm` fotogramas** | Una detección débil y aislada suele ser un FP; la repetición temporal es evidencia |
| **R4 No-fauna** | Clases como `car` se registran pero **no cuentan como fauna** | Evita inflar la riqueza de especies |
| **R5 Evento independiente** | Para cada cámara y especie, registros separados por > `event_gap_min` (30 min por defecto, convención habitual en ecología con cámaras trampa) son eventos distintos | Un mismo animal que se queda frente a la cámara no debe contarse 40 veces |

### 9.2 Salidas
* **Resumen por cámara:** fotogramas, con fauna, vacíos, % vacías, especies distintas, detecciones, eventos, rango de fechas.
* **Actividad por cámara × especie × periodo:** fotogramas con la especie, detecciones (individuos), eventos, máx. individuos en una foto, confianza media. Periodo configurable: `dia`, `semana`, `mes`, `hora`, `franja`.
* **Validación** contra el reporte construido con las **etiquetas reales** (error absoluto medio de conteos y F1 de presencia de especie por fotograma).

> ⚠️ Las marcas de tiempo de CCT pueden tener errores del reloj de la cámara; los patrones horarios son ilustrativos.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class MonitorConfig:
    conf_default: float = 0.25                           # R1
    conf_by_species: dict = field(default_factory=dict)  # R1: p. ej. {'bobcat': 0.15}
    min_rel_area: float = 0.0005                         # R2
    conf_confirm: float = 0.60                           # R3
    min_frames_confirm: int = 2                          # R3
    burst_gap_s: float = 10.0                            # R3 (si no hay seq_id)
    non_animal: tuple = ('car',)                         # R4
    event_gap_min: float = 30.0                          # R5

def make_frames(d):
    """Tabla de fotogramas de UNA o varias cámaras: uid, cámara, fecha-hora, ráfaga, tamaño."""
    f = pd.DataFrame(dict(uid=d.uid.values, camera=d.location.values, datetime=pd.to_datetime(d.date.values, errors='coerce'),
                          seq=d.seq_id.values, W=d.W.values, H=d.H.values))
    bad = f.datetime.isna().sum()
    if bad: print(f'⚠️ {bad} fotogramas sin fecha válida: se excluyen del reporte temporal')
    return f.dropna(subset=['datetime']).sort_values(['camera', 'datetime']).reset_index(drop=True)

def add_bursts(f, cfg):
    """Ráfaga = seq_id si existe; si no, agrupa por proximidad temporal."""
    f = f.copy()
    if f.seq.notna().all() and (f.seq != 'None').all():
        f['burst'] = f.camera.astype(str) + '|' + f.seq.astype(str)
    else:
        gap = f.groupby('camera').datetime.diff().dt.total_seconds()
        f['burst'] = f.camera.astype(str) + '|' + ((gap.isna() | (gap > cfg.burst_gap_s)).groupby(f.camera).cumsum()).astype(str)
    return f

def apply_rules(dets_arr, frames, cfg):
    """dets_arr: Nx7 [uid,x,y,w,h,conf,clase+1]. Devuelve detecciones CONFIRMADAS con columna is_fauna."""
    f = add_bursts(frames, cfg)
    d = pd.DataFrame(dets_arr, columns=['uid', 'x', 'y', 'w', 'h', 'conf', 'cls']); d['uid'] = d.uid.astype(int); d['cls'] = d.cls.astype(int) - 1
    d['species'] = d.cls.map(dict(enumerate(class_names)))
    d = d.merge(f[['uid', 'camera', 'datetime', 'burst', 'W', 'H']], on='uid', how='inner')
    thr = d.species.map(cfg.conf_by_species).fillna(cfg.conf_default)
    d = d[(d.conf >= thr) & (d.w * d.h / (d.W * d.H) >= cfg.min_rel_area)]                 # R1 + R2
    g = d.groupby(['burst', 'species']).agg(max_conf=('conf', 'max'), n_frames=('uid', 'nunique')).reset_index()
    g['ok'] = (g.max_conf >= cfg.conf_confirm) | (g.n_frames >= cfg.min_frames_confirm)    # R3
    d = d.merge(g[['burst', 'species', 'ok']], on=['burst', 'species']); d = d[d.ok].drop(columns='ok')
    d['is_fauna'] = ~d.species.isin(cfg.non_animal)                                         # R4
    return d.reset_index(drop=True)

def assign_period(dt, mode):
    if mode == 'dia':     return dt.dt.strftime('%Y-%m-%d')
    if mode == 'semana':  return dt.dt.strftime('%G-S%V')
    if mode == 'mes':     return dt.dt.strftime('%Y-%m')
    if mode == 'hora':    return dt.dt.hour.map(lambda h: f'{h:02d}h')
    if mode == 'franja':  return pd.cut(dt.dt.hour, [-1, 5, 11, 17, 23], labels=['1) 00-06 madrugada', '2) 06-12 mañana', '3) 12-18 tarde', '4) 18-24 noche']).astype(str)
    raise ValueError(mode)

def build_reports(frames, dets, cfg, period='franja'):
    fa = dets[dets.is_fauna].reset_index(drop=True)
    f = frames.copy()
    pf = fa.groupby('uid').size().rename('n_individuos')
    f = f.merge(pf, on='uid', how='left'); f['n_individuos'] = f.n_individuos.fillna(0).astype(int); f['con_fauna'] = f.n_individuos > 0
    # R5: eventos independientes
    ev = fa.sort_values(['camera', 'species', 'datetime']).reset_index(drop=True)
    gap = ev.groupby(['camera', 'species']).datetime.diff().dt.total_seconds() / 60
    ev['evento'] = (gap.isna() | (gap > cfg.event_gap_min)).astype(int).groupby([ev.camera, ev.species]).cumsum()
    ev['periodo'] = assign_period(ev.datetime, period)
    # resumen por cámara
    cam = f.groupby('camera').agg(fotogramas=('uid', 'size'), con_fauna=('con_fauna', 'sum'), inicio=('datetime', 'min'), fin=('datetime', 'max'))
    cam['vacias'] = cam.fotogramas - cam.con_fauna
    cam['pct_vacias'] = (100 * cam.vacias / cam.fotogramas).round(1); cam['pct_con_fauna'] = (100 - cam.pct_vacias).round(1)
    cam['especies_distintas'] = fa.groupby('camera').species.nunique(); cam['detecciones'] = fa.groupby('camera').size()
    cam['eventos'] = ev.drop_duplicates(['camera', 'species', 'evento']).groupby('camera').size()
    cam = cam.fillna({'especies_distintas': 0, 'detecciones': 0, 'eventos': 0}).astype({'especies_distintas': int, 'detecciones': int, 'eventos': int})
    # actividad por cámara × especie × periodo
    if len(ev):
        cnt = ev.groupby(['camera', 'species', 'periodo', 'uid']).size()
        act = ev.groupby(['camera', 'species', 'periodo']).agg(fotogramas=('uid', 'nunique'), detecciones=('uid', 'size'),
                                                                eventos=('evento', 'nunique'), conf_media=('conf', 'mean'))
        act['max_individuos_por_foto'] = cnt.groupby(level=[0, 1, 2]).max()
        act = act.round({'conf_media': 3}).reset_index()
    else:
        act = pd.DataFrame(columns=['camera', 'species', 'periodo', 'fotogramas', 'detecciones', 'eventos', 'conf_media', 'max_individuos_por_foto'])
    return dict(camara=cam.reset_index(), actividad=act, fotogramas=f, eventos=ev)

def gt_as_dets(d):
    """Detecciones 'perfectas' construidas con las etiquetas reales (conf=1) para obtener el reporte verdadero."""
    rows = [(r.uid, x, y, w, h, 1.0, c + 1) for r in d.itertuples() for (c, x, y, w, h) in r.boxes]
    return np.array(rows, dtype=float).reshape(-1, 7)

### 9.3 Calibración de los umbrales con **validación** (no con test)
**Qué hace:** prueba una pequeña rejilla de umbrales (R1, R3) sobre `trans_val` (una cámara no vista en entrenamiento, distinta de las de test) y elige la combinación con mejor **F1 de presencia de especie por fotograma**. Así los parámetros del módulo no se afinan mirando la cámara de la demostración (regla anti-fuga A.6-3).

In [ ]:
VAL = ds.df[ds.df.split == 'trans_val'].reset_index(drop=True)
DETS_VAL = predict_split(BEST, VAL, 'E0_trans_val')
frames_val = make_frames(VAL)

def presence_set(dets_df, uids):
    s = dets_df[dets_df.is_fauna]; return set(zip(s.uid.astype(int), s.species))
gt_val = apply_rules(gt_as_dets(VAL), frames_val, MonitorConfig(conf_default=0.0, conf_confirm=0.0, min_rel_area=0.0))
truth = presence_set(gt_val, frames_val.uid)

rows = []
for cd in [0.15, 0.25, 0.35, 0.50]:
    for cc in [0.50, 0.60, 0.80]:
        for mf in [1, 2]:
            cfg = MonitorConfig(conf_default=cd, conf_confirm=cc, min_frames_confirm=mf)
            pred = presence_set(apply_rules(DETS_VAL, frames_val, cfg), frames_val.uid)
            tp = len(pred & truth); p = tp / max(1, len(pred)); r = tp / max(1, len(truth))
            rows.append(dict(conf_default=cd, conf_confirm=cc, min_frames=mf, precision=p, recall=r, F1=2 * p * r / max(1e-9, p + r)))
grid = pd.DataFrame(rows).sort_values('F1', ascending=False).round(3); grid.to_csv(TABLES / '18_calibracion_umbrales_val.csv', index=False)
display(grid.head(8))
best = grid.iloc[0]
CFG_MON = MonitorConfig(conf_default=float(best.conf_default), conf_confirm=float(best.conf_confirm), min_frames_confirm=int(best.min_frames))
print('Configuración elegida con VALIDACIÓN:', CFG_MON)

### 9.4 Demostración en una cámara **no vista** en entrenamiento
Se elige la ubicación de `trans_test` con más fotogramas (una cámara que el modelo nunca vio), se ejecutan las reglas sobre **toda su secuencia temporal** y se genera el reporte tabular (CSV + Excel en Drive). Además se compara con el reporte “verdadero” construido con las etiquetas, para cuantificar **qué tan fiable es el reporte automático**.

In [ ]:
cand = ds.df[ds.df.split == 'trans_test'].groupby('location').size().sort_values(ascending=False)
DEMO_LOC = cand.index[0]
d_demo = ds.df[(ds.df.split == 'trans_test') & (ds.df.location == DEMO_LOC)].reset_index(drop=True)
print(f'Cámara de demostración: {DEMO_LOC} ({len(d_demo)} fotogramas) — NO aparece en entrenamiento: {DEMO_LOC not in set(ds.df[ds.df.split=="train"].location)}')

frames = make_frames(d_demo)
dets_demo = DETS['trans_test'][np.isin(DETS['trans_test'][:, 0], d_demo.uid.values)]     # predicciones del modelo sobre esa cámara
conf_dets = apply_rules(dets_demo, frames, CFG_MON)
REP = build_reports(frames, conf_dets, CFG_MON, period='franja')

print('\n=== Resumen por cámara ==='); display(REP['camara'])
print('\n=== Actividad por especie y periodo (franja del día) ==='); display(REP['actividad'])

# Periodo diario y por hora, para gráficos
REP_D = build_reports(frames, conf_dets, CFG_MON, period='dia'); REP_H = build_reports(frames, conf_dets, CFG_MON, period='hora')
with pd.ExcelWriter(OUT / 'reporte_monitoreo.xlsx', engine='openpyxl') as xw:
    REP['camara'].to_excel(xw, sheet_name='resumen_camara', index=False)
    REP['actividad'].to_excel(xw, sheet_name='actividad_franja', index=False)
    REP_H['actividad'].to_excel(xw, sheet_name='actividad_hora', index=False)
    REP_D['actividad'].to_excel(xw, sheet_name='actividad_dia', index=False)
    REP['eventos'].to_excel(xw, sheet_name='detecciones_confirmadas', index=False)
REP['camara'].to_csv(TABLES / '19_monitoreo_resumen_camara.csv', index=False)
REP['actividad'].to_csv(TABLES / '20_monitoreo_actividad_franja.csv', index=False)
REP_H['actividad'].to_csv(TABLES / '21_monitoreo_actividad_hora.csv', index=False)
print('Guardado:', OUT / 'reporte_monitoreo.xlsx')

In [ ]:
# Mapa de calor especie × hora y línea de tiempo diaria
h = REP_H['actividad'].pivot_table(index='species', columns='periodo', values='detecciones', aggfunc='sum', fill_value=0)
h = h.reindex(columns=[f'{i:02d}h' for i in range(24)], fill_value=0)
dd = REP_D['fotogramas'].assign(dia=lambda x: x.datetime.dt.date).groupby('dia').agg(fotos=('uid', 'size'), con_fauna=('con_fauna', 'sum'))
fig, ax = plt.subplots(1, 2, figsize=(17, 4.5), gridspec_kw={'width_ratios': [1.3, 1]})
if len(h):
    im = ax[0].imshow(h.values, aspect='auto', cmap='YlOrRd'); ax[0].set_yticks(range(len(h))); ax[0].set_yticklabels(h.index)
    ax[0].set_xticks(range(24)); ax[0].set_xticklabels(range(24)); ax[0].set_xlabel('hora del día'); plt.colorbar(im, ax=ax[0], label='detecciones')
ax[0].set_title(f'Cámara {DEMO_LOC}: actividad por especie y hora (reporte automático)')
ax[1].plot(dd.index, dd.fotos, label='fotogramas'); ax[1].plot(dd.index, dd.con_fauna, label='con fauna'); ax[1].legend()
ax[1].set_title('Fotogramas por día'); plt.setp(ax[1].get_xticklabels(), rotation=45)
plt.tight_layout(); plt.savefig(FIGS / '10_monitoreo_demo.png', dpi=130); plt.show()

### 9.5 ¿Qué tan fiable es el reporte automático? (validación contra etiquetas reales)
Se construye el mismo reporte con las etiquetas reales y se compara: error absoluto medio (MAE) en **detecciones** y **eventos** por especie y franja, diferencia de % de fotogramas vacíos, y F1 de presencia de especie por fotograma. Es la evidencia de cuánto puede confiarse en el reporte (¡y en qué especies no!).

In [ ]:
gt_conf = apply_rules(gt_as_dets(d_demo), frames, MonitorConfig(conf_default=0.0, conf_confirm=0.0, min_rel_area=0.0, non_animal=CFG.non_animal))
REP_GT = build_reports(frames, gt_conf, CFG_MON, period='franja')

m = REP['actividad'].merge(REP_GT['actividad'], on=['camera', 'species', 'periodo'], how='outer', suffixes=('_modelo', '_real')).fillna(0)
m['err_detecciones'] = (m.detecciones_modelo - m.detecciones_real).abs(); m['err_eventos'] = (m.eventos_modelo - m.eventos_real).abs()
print(f'MAE detecciones por (especie, franja): {m.err_detecciones.mean():.2f} | MAE eventos: {m.err_eventos.mean():.2f}')
tot = m.groupby('species')[['detecciones_modelo', 'detecciones_real', 'eventos_modelo', 'eventos_real']].sum()
tot['err_rel_detecciones_%'] = (100 * (tot.detecciones_modelo - tot.detecciones_real) / tot.detecciones_real.replace(0, np.nan)).round(1)
display(tot.sort_values('detecciones_real', ascending=False))
tot.to_csv(TABLES / '22_monitoreo_modelo_vs_real_por_especie.csv')

pc = REP['camara'].merge(REP_GT['camara'], on='camera', suffixes=('_modelo', '_real'))
display(pc[['camera', 'pct_vacias_modelo', 'pct_vacias_real', 'especies_distintas_modelo', 'especies_distintas_real', 'eventos_modelo', 'eventos_real']])

pred_set = set(zip(conf_dets[conf_dets.is_fauna].uid.astype(int), conf_dets[conf_dets.is_fauna].species))
tru_set = set(zip(gt_conf[gt_conf.is_fauna].uid.astype(int), gt_conf[gt_conf.is_fauna].species))
tp = len(pred_set & tru_set); p = tp / max(1, len(pred_set)); r = tp / max(1, len(tru_set))
print(f'Presencia de especie por fotograma → precisión {p:.3f} · recall {r:.3f} · F1 {2*p*r/max(1e-9,p+r):.3f}')

## 10. Discusión: uso responsable de IA en conservación (Entregable 6)

> *(Texto base de media página; adáptalo con tus cifras: tasa de FP en vacías, recall de “hay fauna”, brecha cis→trans, AP de las especies raras.)*

**Asimetría de costos.** Un **falso negativo** (el modelo no detecta una especie amenazada) es silencioso: nadie revisa esa foto, la especie desaparece de los conteos de ocupación y abundancia, y las decisiones de manejo (zonas de protección, patrullaje, monitoreo de caza) se toman con una imagen incompleta. Un error así puede ser irreversible si la población decae sin que se advierta. Un **falso positivo**, en cambio, cuesta tiempo de un especialista que revisa una detección incorrecta: es un costo acotado, visible y recuperable. Por eso el umbral de decisión no debería ser el mismo para todas las especies: conviene **bajar el umbral (más sensibilidad)** para especies amenazadas o raras y aceptar más revisión humana (en este cuaderno, `conf_by_species` en `MonitorConfig`), y mantener umbrales más exigentes para especies abundantes donde un FP no cambia ninguna decisión. Además, las especies raras son justamente las peor representadas en el entrenamiento (sección 6): su AP es el más bajo y el más incierto, de modo que el riesgo de FN se concentra donde más importa. La salida del sistema debe entenderse como **priorización para revisión humana**, no como censo automático: conservar el flujo *humano en el circuito* (muestreo de fotos “vacías” para auditar FN, revisión de toda detección de especies de alto interés) y reportar incertidumbre.

**Despliegue en otra región sin validación local.** Este modelo se entrenó con fauna del suroeste de EE. UU. Llevarlo a la Amazonía o los Andes peruanos sin validarlo es riesgoso por tres razones: (1) **especies que no existen en el entrenamiento** (jaguar, oso andino, tapir…) serán forzadas a alguna clase conocida —o ignoradas— con confianza engañosa; (2) **cambio de dominio**: vegetación densa, humedad, otra luz, otras alturas de cámara y distancias; nuestro experimento cis/trans ya muestra que incluso *dentro* del mismo dataset un fondo nuevo degrada el desempeño, y el salto entre regiones es mayor; (3) las **métricas globales ocultan** fallos por especie y por cámara. Antes de usarlo se requiere una **validación local**: un conjunto etiquetado por expertos de las cámaras reales, métricas por especie y por cámara, calibración de umbrales con ese conjunto, auditoría periódica y, si procede, *fine-tuning* con datos locales. Y deben documentarse sus límites (qué especies conoce, con qué fondos) para que los resultados no se interpreten más allá de lo que respalda la evidencia.

## 11. Cierre: checklist de entregables y empaquetado

La celda siguiente lista lo generado y lo comprime en un `.zip` en Drive. Verifica que cada entregable del documento tiene su evidencia:

| # | Entregable | Evidencia generada |
|---|---|---|
| 1 | Modelo y decisiones | `runs/E0_base/weights/best.pt`, `decisiones_entrenamiento.json`, `args.yaml`, curvas (`03_…png`) |
| 2 | mAP global y por especie, desbalance | `03_`, `04_`, `05_` (CSV) y `04_desbalance_vs_ap.png`, matriz de confusión |
| 3 | Generalización cis vs trans | `07_`–`13_` (CSV), `05_`–`07_…png` (por ubicación, Grad-CAM) |
| 4 | Oclusión y tamaño | `14_`–`17_` (CSV), `08_ejemplos_oclusion.png`, `09_robustez.png` |
| 5 | Monitoreo + demo | `reporte_monitoreo.xlsx`, `18_`–`22_` (CSV), `10_monitoreo_demo.png` |
| 6 | Discusión ética | Sección 10 (adáptala con tus cifras) |

In [ ]:
files = sorted([p for p in OUT.rglob('*') if p.is_file() and p.suffix in {'.csv', '.png', '.xlsx', '.json', '.yaml'} and 'preds' not in p.parts])
print(f'{len(files)} archivos de resultados:'); [print(' ·', p.relative_to(OUT)) for p in files[:80]]
zpath = OUT / 'entregables_resultados.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files: z.write(p, p.relative_to(OUT))
    if (RUNS / 'E0_base' / 'weights' / 'best.pt').exists(): z.write(RUNS / 'E0_base' / 'weights' / 'best.pt', 'E0_best.pt')
print('\nPaquete:', zpath, f'({zpath.stat().st_size/1e6:.1f} MB)')

## 12. Extensión opcional: generalización entre datasets (otra región)
Si tu grupo quiere sumar el experimento *entre-datasets*:
1. Elige en https://lila.science/datasets un dataset de **otra región** con cajas y formato COCO Camera Traps; revisa su licencia.
2. Mapea sus especies a las de CCT-20 (las que coincidan) y marca el resto como “no vistas”.
3. Reutiliza `load_coco_split` / `build_dataset` (adaptando rutas) y **evalúa sin reentrenar** (`ultra_val` + `coco_eval`).
4. Reporta: caída respecto a `trans_test`, falsas alarmas en vacías y **especies del nuevo dataset que el modelo no puede nombrar**. Es el experimento más cercano a un despliegue en la Amazonía/Andes.